**Gravitational Clustering**

Compute barycenter trajectories (imports and exports separately).

Compute derived features: velocity, heading, displacement, curvature, break-period shifts.

Define a trajectory distance (synchronized + directional) and cluster countries.

For each cluster, compute a fixed “attractor” as a prototype (endpoint-based or direction-based).

Validate stability (cluster robustness over time windows / bootstrap years) and finalize.

1. **Barycenters for all countries**

In [24]:
import os
import pandas as pd
import numpy as np
import duckdb

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
TRADE_DIR = r"C:\Python\trade\dataverse_files"
CENTROID_FILE = r"C:\Python\trade\geo\country_centroids_augmented.csv"
OUT_DIR = r"C:\Python\trade\geo\gravitationalcluster"

os.makedirs(OUT_DIR, exist_ok=True)

START_YEAR = 1977
END_YEAR = 2022

# ------------------------------------------------------------
# LOAD CENTROIDS (authoritative)
# ------------------------------------------------------------
centroids = pd.read_csv(CENTROID_FILE)

# required columns
assert {"code", "lat", "lon"}.issubset(centroids.columns)

centroids = centroids[["code", "lat", "lon"]].dropna()
centroids = centroids.set_index("code")

# ------------------------------------------------------------
# DUCKDB CONNECTION
# ------------------------------------------------------------
con = duckdb.connect()

# container for results (append then split by country)
rows = []

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
for year in range(START_YEAR, END_YEAR + 1):
    path = os.path.join(TRADE_DIR, f"S2_{year}.parquet")
    if not os.path.exists(path):
        continue

    print(f"Processing {year}")

    df = con.execute(f"""
        SELECT exporter, importer, value_final
        FROM read_parquet('{path}')
        WHERE value_final IS NOT NULL
          AND value_final > 0
    """).df()

    # --------------------------------------------------------
    # EXPORTS: country i → partners j
    # --------------------------------------------------------
    exp = (
        df
        .merge(centroids, left_on="importer", right_index=True, how="inner")
        .groupby("exporter", as_index=False)
        .apply(lambda x: pd.Series({
            "lat_exports": np.average(x["lat"], weights=x["value_final"]),
            "lon_exports": np.average(x["lon"], weights=x["value_final"]),
            "exports_weight": x["value_final"].sum()
        }))
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # IMPORTS: partners j → country i
    # --------------------------------------------------------
    imp = (
        df
        .merge(centroids, left_on="exporter", right_index=True, how="inner")
        .groupby("importer", as_index=False)
        .apply(lambda x: pd.Series({
            "lat_imports": np.average(x["lat"], weights=x["value_final"]),
            "lon_imports": np.average(x["lon"], weights=x["value_final"]),
            "imports_weight": x["value_final"].sum()
        }))
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # COMBINE
    # --------------------------------------------------------
    year_df = (
        exp.merge(
            imp,
            left_on="exporter",
            right_on="importer",
            how="outer"
        )
        .rename(columns={"exporter": "country"})
        .drop(columns=["importer"])
    )

    year_df["year"] = year
    rows.append(year_df)

# ------------------------------------------------------------
# FINAL DATASET
# ------------------------------------------------------------
bary = pd.concat(rows, ignore_index=True)

# ------------------------------------------------------------
# WRITE ONE FILE PER COUNTRY
# ------------------------------------------------------------
for country, g in bary.groupby("country"):
    out = os.path.join(
        OUT_DIR,
        f"barycenter_{country}_1977_2022.csv"
    )
    g.sort_values("year").to_csv(out, index=False)

print("✓ Barycenters computed for all countries")
print(f"✓ Output directory: {OUT_DIR}")


Processing 1977
Processing 1978
Processing 1979
Processing 1980
Processing 1981
Processing 1982
Processing 1983
Processing 1984
Processing 1985
Processing 1986
Processing 1987
Processing 1988
Processing 1989
Processing 1990
Processing 1991
Processing 1992
Processing 1993
Processing 1994
Processing 1995
Processing 1996
Processing 1997
Processing 1998
Processing 1999
Processing 2000
Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
✓ Barycenters computed for all countries
✓ Output directory: C:\Python\trade\geo\gravitationalcluster


2. Compute derived features: velocity, heading, displacement, curvature, break-period shifts.

In [25]:
import os
import glob
import math
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

# Input pattern: one file per country
IN_GLOB = os.path.join(BASE_DIR, "barycenter_*_1977_2022.csv")

# Outputs
OUT_STATE_LONG   = os.path.join(BASE_DIR, "gc_state_long.csv")
OUT_STEPS_LONG   = os.path.join(BASE_DIR, "gc_steps_long.csv")
OUT_FEATURES     = os.path.join(BASE_DIR, "gc_features_country_flow.csv")

# Earth radius for km conversions (optional)
EARTH_RADIUS_KM = 6371.0088

# Minimum observations for period mean position
MIN_OBS_PER_PERIOD = 2

# Break-period definitions (inclusive bounds)
# You can change these freely; code uses whatever years exist.
PERIODS = [
    ("P1_1977_1988", 1977, 1988),
    ("P2_1989_2001", 1989, 2001),
    ("P3_2002_2009", 2002, 2009),
    ("P4_2010_2018", 2010, 2018),
    ("P5_2019_2022", 2019, 2022),
]

# ============================================================
# GEOMETRY HELPERS (sphere-safe)
# ============================================================
def deg2rad(x):
    return np.deg2rad(x)

def wrap_pi(angle_rad):
    """Wrap to [-pi, pi]."""
    return (angle_rad + np.pi) % (2 * np.pi) - np.pi

def latlon_to_unitvec(lat_deg, lon_deg):
    """lat/lon in degrees -> unit vector (x,y,z)."""
    lat = deg2rad(lat_deg)
    lon = deg2rad(lon_deg)
    clat = np.cos(lat)
    x = clat * np.cos(lon)
    y = clat * np.sin(lon)
    z = np.sin(lat)
    return x, y, z

def gc_distance_rad_from_dot(dot):
    """Great-circle distance in radians from dot product, with clipping."""
    return np.arccos(np.clip(dot, -1.0, 1.0))

def initial_bearing_deg(lat1_deg, lon1_deg, lat2_deg, lon2_deg):
    """
    Initial bearing from point 1 to point 2.
    Returns bearing in degrees [0, 360).
    """
    lat1 = deg2rad(lat1_deg)
    lat2 = deg2rad(lat2_deg)
    dlon = deg2rad(lon2_deg - lon1_deg)

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    brng = np.arctan2(y, x)  # [-pi, pi]
    brng_deg = (np.rad2deg(brng) + 360.0) % 360.0
    return brng_deg

def circ_mean_deg(angles_deg):
    """Circular mean for degrees in [0,360)."""
    a = np.deg2rad(np.asarray(angles_deg, dtype=float))
    a = a[~np.isnan(a)]
    if a.size == 0:
        return np.nan
    s = np.mean(np.sin(a))
    c = np.mean(np.cos(a))
    mu = np.arctan2(s, c)
    return (np.rad2deg(mu) + 360.0) % 360.0

def circ_var_unit(angles_deg):
    """Circular variance in [0,1], using mean resultant length."""
    a = np.deg2rad(np.asarray(angles_deg, dtype=float))
    a = a[~np.isnan(a)]
    if a.size == 0:
        return np.nan
    R = np.hypot(np.mean(np.cos(a)), np.mean(np.sin(a)))
    return 1.0 - R

def spherical_mean_unitvec(xyz):
    """
    Spherical mean of unit vectors.
    xyz: array shape (n,3)
    Returns unit vector (3,) or (nan,nan,nan) if undefined.
    """
    xyz = np.asarray(xyz, dtype=float)
    xyz = xyz[~np.isnan(xyz).any(axis=1)]
    if xyz.shape[0] == 0:
        return np.array([np.nan, np.nan, np.nan])
    v = xyz.sum(axis=0)
    norm = np.linalg.norm(v)
    if norm == 0 or np.isnan(norm):
        return np.array([np.nan, np.nan, np.nan])
    return v / norm

# ============================================================
# I/O + FEATURE PIPELINE
# ============================================================
def parse_country_from_filename(fp):
    # barycenter_<ISO3>_1977_2022.csv
    base = os.path.basename(fp)
    parts = base.split("_")
    if len(parts) < 2:
        return None
    return parts[1]

def flow_specs():
    # We compute separately for exports and imports
    return [
        ("exports", "lat_exports", "lon_exports", "exports_weight"),
        ("imports", "lat_imports", "lon_imports", "imports_weight"),
    ]

def build_state_long_for_country(country, df):
    """
    Build long state table for this country for both flows.
    """
    out_rows = []
    for flow, latc, lonc, wc in flow_specs():
        if latc not in df.columns or lonc not in df.columns:
            continue

        sub = df[["year", latc, lonc] + ([wc] if wc in df.columns else [])].copy()
        sub = sub.dropna(subset=["year", latc, lonc])
        if sub.empty:
            continue

        sub["country"] = country
        sub["flow"] = flow
        sub = sub.rename(columns={latc: "lat", lonc: "lon"})
        if wc in sub.columns:
            sub = sub.rename(columns={wc: "weight"})
        else:
            sub["weight"] = np.nan

        x, y, z = latlon_to_unitvec(sub["lat"].astype(float), sub["lon"].astype(float))
        sub["x"] = x
        sub["y"] = y
        sub["z"] = z

        # Keep canonical order
        sub = sub[["country", "flow", "year", "lat", "lon", "x", "y", "z", "weight"]].sort_values("year")
        out_rows.append(sub)

    if out_rows:
        return pd.concat(out_rows, ignore_index=True)
    return pd.DataFrame(columns=["country","flow","year","lat","lon","x","y","z","weight"])

def build_steps_for_country_flow(state_sub):
    """
    state_sub: rows for a single (country, flow) sorted by year, with x,y,z,lat,lon.
    Creates step records using consecutive available years (not necessarily year+1).
    """
    state_sub = state_sub.sort_values("year").reset_index(drop=True)
    n = len(state_sub)
    if n < 2:
        return pd.DataFrame(columns=[
            "country","flow","year_from","year_to","dt_years",
            "dist_rad","dist_km","velocity_rad_per_year","velocity_km_per_year",
            "bearing_deg","turning_angle_deg"
        ])

    rows = []
    bearings = []

    for idx in range(n - 1):
        r1 = state_sub.iloc[idx]
        r2 = state_sub.iloc[idx + 1]
        y1, y2 = int(r1["year"]), int(r2["year"])
        dt = y2 - y1
        if dt <= 0:
            continue

        dot = float(r1["x"] * r2["x"] + r1["y"] * r2["y"] + r1["z"] * r2["z"])
        dist_rad = float(gc_distance_rad_from_dot(dot))
        dist_km = dist_rad * EARTH_RADIUS_KM

        v_rad = dist_rad / dt
        v_km = dist_km / dt

        brng = float(initial_bearing_deg(r1["lat"], r1["lon"], r2["lat"], r2["lon"]))
        bearings.append(brng)

        rows.append({
            "country": r1["country"],
            "flow": r1["flow"],
            "year_from": y1,
            "year_to": y2,
            "dt_years": dt,
            "dist_rad": dist_rad,
            "dist_km": dist_km,
            "velocity_rad_per_year": v_rad,
            "velocity_km_per_year": v_km,
            "bearing_deg": brng,
            # turning angle filled after we have bearings
            "turning_angle_deg": np.nan,
        })

    steps = pd.DataFrame(rows)
    if steps.empty:
        return steps

    # Turning angle for consecutive bearings (requires 3 points -> 2 steps)
    # turning angle at step j (from step j-1 to step j) assigned to row j
    b = steps["bearing_deg"].to_numpy(dtype=float)
    if b.size >= 2:
        b_rad = np.deg2rad(b)
        turn = wrap_pi(b_rad[1:] - b_rad[:-1])
        steps.loc[steps.index[1:], "turning_angle_deg"] = np.rad2deg(turn)

    return steps

def build_country_flow_features(state_sub, steps_sub):
    """
    state_sub: (country, flow) states
    steps_sub: (country, flow) steps
    Returns one-row dataframe with summary + period stats.
    """
    country = state_sub["country"].iloc[0]
    flow = state_sub["flow"].iloc[0]

    years = state_sub["year"].astype(int).to_numpy()
    n_obs = len(years)
    start_year = int(years.min()) if n_obs else np.nan
    end_year = int(years.max()) if n_obs else np.nan

    # Net displacement (start to end)
    if n_obs >= 2:
        x0 = state_sub.loc[state_sub["year"] == start_year, ["x","y","z"]].iloc[0].to_numpy(dtype=float)
        xT = state_sub.loc[state_sub["year"] == end_year, ["x","y","z"]].iloc[0].to_numpy(dtype=float)
        D_rad = float(gc_distance_rad_from_dot(float(np.dot(x0, xT))))
        D_km = D_rad * EARTH_RADIUS_KM
    else:
        D_rad = np.nan
        D_km = np.nan

    # Path length
    if not steps_sub.empty:
        L_rad = float(steps_sub["dist_rad"].sum())
        L_km = float(steps_sub["dist_km"].sum())
        mean_v_km = float(steps_sub["velocity_km_per_year"].mean())
        std_v_km = float(steps_sub["velocity_km_per_year"].std(ddof=0))
        bearing_mean = float(circ_mean_deg(steps_sub["bearing_deg"]))
        bearing_var = float(circ_var_unit(steps_sub["bearing_deg"]))
        mean_abs_turn = float(np.nanmean(np.abs(steps_sub["turning_angle_deg"])))
    else:
        L_rad = np.nan
        L_km = np.nan
        mean_v_km = np.nan
        std_v_km = np.nan
        bearing_mean = np.nan
        bearing_var = np.nan
        mean_abs_turn = np.nan

    straightness = (D_rad / L_rad) if (L_rad is not None and not np.isnan(L_rad) and L_rad > 0 and not np.isnan(D_rad)) else np.nan

    # Period means (spherical mean of positions in each period) + shifts between adjacent periods
    period_vecs = {}
    period_n = {}
    for pname, a, b in PERIODS:
        mask = (state_sub["year"] >= a) & (state_sub["year"] <= b)
        seg = state_sub.loc[mask, ["x","y","z"]].to_numpy(dtype=float)
        seg = seg[~np.isnan(seg).any(axis=1)]
        period_n[pname] = int(seg.shape[0])
        if seg.shape[0] >= MIN_OBS_PER_PERIOD:
            period_vecs[pname] = spherical_mean_unitvec(seg)
        else:
            period_vecs[pname] = np.array([np.nan, np.nan, np.nan])

    # Shifts between adjacent periods (great-circle angle between period mean vectors)
    period_shift = {}
    for idx in range(len(PERIODS) - 1):
        p1 = PERIODS[idx][0]
        p2 = PERIODS[idx + 1][0]
        v1 = period_vecs[p1]
        v2 = period_vecs[p2]
        if np.isnan(v1).any() or np.isnan(v2).any():
            period_shift[f"shift_{p1}_to_{p2}_km"] = np.nan
            period_shift[f"shift_{p1}_to_{p2}_rad"] = np.nan
        else:
            ang_rad = float(gc_distance_rad_from_dot(float(np.dot(v1, v2))))
            period_shift[f"shift_{p1}_to_{p2}_rad"] = ang_rad
            period_shift[f"shift_{p1}_to_{p2}_km"] = ang_rad * EARTH_RADIUS_KM

    row = {
        "country": country,
        "flow": flow,
        "n_obs": n_obs,
        "start_year": start_year,
        "end_year": end_year,
        "net_disp_rad": D_rad,
        "net_disp_km": D_km,
        "path_len_rad": L_rad,
        "path_len_km": L_km,
        "straightness": straightness,
        "mean_speed_km_per_year": mean_v_km,
        "std_speed_km_per_year": std_v_km,
        "bearing_mean_deg": bearing_mean,
        "bearing_circ_var": bearing_var,
        "mean_abs_turn_deg": mean_abs_turn,
    }

    # Add period coverage + mean vectors (optional but useful)
    for pname, _, _ in PERIODS:
        row[f"{pname}_nobs"] = period_n[pname]
        v = period_vecs[pname]
        row[f"{pname}_x"] = v[0]
        row[f"{pname}_y"] = v[1]
        row[f"{pname}_z"] = v[2]

    row.update(period_shift)
    return pd.DataFrame([row])

# ============================================================
# MAIN
# ============================================================
files = sorted(glob.glob(IN_GLOB))
if not files:
    raise FileNotFoundError(f"No input files found matching: {IN_GLOB}")

all_state = []
all_steps = []
all_features = []

for fp in files:
    country = parse_country_from_filename(fp)
    if not country:
        continue

    df = pd.read_csv(fp)
    if "year" not in df.columns:
        continue

    # Build states (exports + imports) for this country
    state = build_state_long_for_country(country, df)
    if state.empty:
        continue
    all_state.append(state)

    # Steps + features per flow
    for flow in ["exports", "imports"]:
        ssub = state[state["flow"] == flow].sort_values("year").reset_index(drop=True)
        if len(ssub) == 0:
            continue

        steps = build_steps_for_country_flow(ssub)
        if not steps.empty:
            all_steps.append(steps)

        feats = build_country_flow_features(ssub, steps if not steps.empty else pd.DataFrame())
        all_features.append(feats)

# Concatenate + write
state_long = pd.concat(all_state, ignore_index=True)
steps_long = pd.concat(all_steps, ignore_index=True) if all_steps else pd.DataFrame()
features = pd.concat(all_features, ignore_index=True)

state_long.to_csv(OUT_STATE_LONG, index=False)
steps_long.to_csv(OUT_STEPS_LONG, index=False)
features.to_csv(OUT_FEATURES, index=False)

print("✓ Wrote:")
print(f"  {OUT_STATE_LONG}   (country-year states, sphere-safe x/y/z)")
print(f"  {OUT_STEPS_LONG}   (country-step dynamics: distance, velocity, bearing, turning)")
print(f"  {OUT_FEATURES}     (country-flow summaries + period shifts)")
print("\nMissing years handling:")
print("  - Steps are computed between consecutive available years (dt can be > 1).")
print("  - Velocities are normalized by dt (per-year).")
print("  - Period means/shifts use whatever years exist; require MIN_OBS_PER_PERIOD observations.")


✓ Wrote:
  C:\Python\trade\geo\gravitationalcluster\gc_state_long.csv   (country-year states, sphere-safe x/y/z)
  C:\Python\trade\geo\gravitationalcluster\gc_steps_long.csv   (country-step dynamics: distance, velocity, bearing, turning)
  C:\Python\trade\geo\gravitationalcluster\gc_features_country_flow.csv     (country-flow summaries + period shifts)

Missing years handling:
  - Steps are computed between consecutive available years (dt can be > 1).
  - Velocities are normalized by dt (per-year).
  - Period means/shifts use whatever years exist; require MIN_OBS_PER_PERIOD observations.


3.**Define a trajectory distance (synchronized + directional) and cluster countries.**

In [26]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [27]:
import os
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

STATE_LONG = os.path.join(BASE_DIR, "gc_state_long.csv")
# (we will derive step directions from state_long; gc_steps_long not required)

OUT_DIR = BASE_DIR  # keep everything together

FLOWS = ["exports", "imports"]

# Distance construction
MIN_OVERLAP_YEARS = 10     # minimum overlapping years to compute pos distance
MIN_OVERLAP_STEPS = 8      # minimum overlapping steps to compute dir distance
ALPHA_POS = 0.7            # d = alpha*d_pos + (1-alpha)*d_dir

# Clustering
K_MIN, K_MAX = 2, 12       # candidates for #clusters (hierarchical)
LINKAGE = "average"        # average linkage works well with generic distances

# Year universe (use what's in the data; typically 1977-2022)
# ============================================================


# ============================================================
# HELPERS
# ============================================================
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def safe_norm(v, axis=-1, keepdims=False):
    return np.linalg.norm(v, axis=axis, keepdims=keepdims)

def build_country_year_arrays(df_flow):
    """
    df_flow columns: country, year, x,y,z
    returns:
      countries (list)
      years (np array sorted)
      X: np array (N, T, 3) with NaNs where missing
      M: mask (N, T) True where present
    """
    years = np.sort(df_flow["year"].unique())
    countries = np.sort(df_flow["country"].unique())

    year_to_idx = {y:i for i,y in enumerate(years)}
    country_to_idx = {c:i for i,c in enumerate(countries)}

    N = len(countries)
    T = len(years)

    X = np.full((N, T, 3), np.nan, dtype=float)
    M = np.zeros((N, T), dtype=bool)

    for r in df_flow.itertuples(index=False):
        i = country_to_idx[r.country]
        t = year_to_idx[r.year]
        X[i, t, 0] = r.x
        X[i, t, 1] = r.y
        X[i, t, 2] = r.z
        M[i, t] = True

    return countries.tolist(), years, X, M

def build_step_unit_vectors(X, M):
    """
    Build unit direction vectors for steps between consecutive *available* years.
    Returns:
      U: (N, T, 3) unit vector at year index t meaning direction from t -> next available year
      MU: (N, T) mask where U defined
      DT: (N, T) dt in years to next available year (float), NaN if undefined
    """
    N, T, _ = X.shape
    U = np.full((N, T, 3), np.nan, dtype=float)
    MU = np.zeros((N, T), dtype=bool)
    DT = np.full((N, T), np.nan, dtype=float)

    for i in range(N):
        idx = np.where(M[i])[0]
        if idx.size < 2:
            continue
        for k in range(idx.size - 1):
            t0 = idx[k]
            t1 = idx[k+1]
            v = X[i, t1] - X[i, t0]  # chord direction
            n = np.linalg.norm(v)
            if not np.isfinite(n) or n == 0:
                continue
            U[i, t0] = v / n
            MU[i, t0] = True
            DT[i, t0] = float(t1 - t0)  # in index units; converted later using years array
    return U, MU, DT

def pairwise_pos_distance(X, M, min_overlap):
    """
    Average great-circle distance (radians) over overlapping years.
    Returns:
      Dpos (N,N) with NaNs where insufficient overlap
      Over (N,N) overlap counts
    """
    N, T, _ = X.shape
    D = np.full((N, N), np.nan, dtype=float)
    Over = np.zeros((N, N), dtype=int)

    # diagonal
    np.fill_diagonal(D, 0.0)
    np.fill_diagonal(Over, M.sum(axis=1).astype(int))

    for i in range(N):
        Mi = M[i]
        Xi = X[i]
        for j in range(i+1, N):
            mask = Mi & M[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Xi[mask], X[j, mask])
            ang = arccos_clip(dots)  # radians
            D[i, j] = D[j, i] = float(np.mean(ang))
    return D, Over

def pairwise_dir_distance(U, MU, min_overlap):
    """
    Average directional mismatch over overlapping step-years:
      d = mean(1 - dot(u_i, u_j))
    Returns:
      Ddir (N,N) with NaNs where insufficient overlap
      Over (N,N) overlap counts
    """
    N, T, _ = U.shape
    D = np.full((N, N), np.nan, dtype=float)
    Over = np.zeros((N, N), dtype=int)

    np.fill_diagonal(D, 0.0)
    np.fill_diagonal(Over, MU.sum(axis=1).astype(int))

    for i in range(N):
        MUi = MU[i]
        Ui = U[i]
        for j in range(i+1, N):
            mask = MUi & MU[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Ui[mask], U[j, mask])
            mismatch = 1.0 - dots
            D[i, j] = D[j, i] = float(np.mean(mismatch))
    return D, Over

def robust_normalize(D):
    """
    Normalize by median of finite off-diagonal values.
    """
    finite = np.isfinite(D)
    # exclude diagonal
    idx = np.where(finite & (~np.eye(D.shape[0], dtype=bool)))
    vals = D[idx]
    if vals.size == 0:
        return D, np.nan
    med = np.median(vals)
    if med <= 0 or not np.isfinite(med):
        return D, med
    return D / med, med

def fill_missing_with_penalty(D, factor=1.5):
    """
    Replace NaNs with a penalty = factor * max_finite.
    """
    D2 = D.copy()
    finite = np.isfinite(D2)
    if not finite.any():
        raise ValueError("Distance matrix has no finite entries.")
    maxv = np.nanmax(D2[finite])
    penalty = factor * maxv if maxv > 0 else 1.0
    D2[~finite] = penalty
    np.fill_diagonal(D2, 0.0)
    return D2, penalty

def run_hierarchical_clustering(D_pre, countries, flow, out_prefix):
    """
    Chooses k by best silhouette over K_MIN..K_MAX, then fits final model.
    Writes labels and silhouette table.
    """
    D_filled, penalty = fill_missing_with_penalty(D_pre, factor=1.5)

    sil_rows = []
    best = {"k": None, "sil": -np.inf, "labels": None}

    for k in range(K_MIN, K_MAX + 1):
        model = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage=LINKAGE
        )
        labels = model.fit_predict(D_filled)

        # silhouette with precomputed distances requires finite matrix
        sil = silhouette_score(D_filled, labels, metric="precomputed")
        sil_rows.append({"flow": flow, "k": k, "silhouette": sil})

        if sil > best["sil"]:
            best = {"k": k, "sil": sil, "labels": labels}

    sil_df = pd.DataFrame(sil_rows).sort_values(["flow", "k"])
    sil_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_silhouette.csv")
    sil_df.to_csv(sil_path, index=False)

    labels_df = pd.DataFrame({
        "country": countries,
        "flow": flow,
        "cluster": best["labels"],
        "k_selected": best["k"],
        "silhouette_selected": best["sil"],
        "missing_penalty_used": penalty,
        "linkage": LINKAGE,
        "alpha_pos": ALPHA_POS,
        "min_overlap_years": MIN_OVERLAP_YEARS,
        "min_overlap_steps": MIN_OVERLAP_STEPS,
    }).sort_values(["cluster", "country"])

    lab_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_clusters.csv")
    labels_df.to_csv(lab_path, index=False)

    print(f"[{flow}] selected k={best['k']} silhouette={best['sil']:.4f}")
    print(f"  wrote: {sil_path}")
    print(f"  wrote: {lab_path}")

    return labels_df, sil_df


# ============================================================
# MAIN
# ============================================================
if not os.path.exists(STATE_LONG):
    raise FileNotFoundError(f"Missing: {STATE_LONG}")

state = pd.read_csv(STATE_LONG)

required = {"country", "flow", "year", "x", "y", "z"}
missing = required - set(state.columns)
if missing:
    raise ValueError(f"gc_state_long.csv missing columns: {sorted(missing)}")

# ensure types
state["year"] = state["year"].astype(int)

for flow in FLOWS:
    df_flow = state[state["flow"] == flow].copy()
    if df_flow.empty:
        print(f"Skipping flow={flow}: no rows.")
        continue

    # Build country-year arrays
    countries, years, X, M = build_country_year_arrays(df_flow)

    # Convert step DT from index units to year units: since years are consecutive ints in practice,
    # index delta == year delta. Still, for correctness, derive dt_years via years array:
    # dt_years = years[t1] - years[t0]. We only need direction overlap here, not dt.
    U, MU, DT_idx = build_step_unit_vectors(X, M)

    # Pairwise distances
    Dpos, Opos = pairwise_pos_distance(X, M, MIN_OVERLAP_YEARS)
    Ddir, Odir = pairwise_dir_distance(U, MU, MIN_OVERLAP_STEPS)

    # Normalize to comparable scale
    Dpos_n, med_pos = robust_normalize(Dpos)
    Ddir_n, med_dir = robust_normalize(Ddir)

    # Combine
    Dcombo = ALPHA_POS * Dpos_n + (1.0 - ALPHA_POS) * Ddir_n

    # Save distance matrices + overlaps
    out_prefix = "gc"

    Dpos_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_Dpos.csv")
    Ddir_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_Ddir.csv")
    Dcmb_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_Dcombo.csv")
    Opos_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_overlap_years.csv")
    Odir_path = os.path.join(OUT_DIR, f"{out_prefix}_{flow}_overlap_steps.csv")

    pd.DataFrame(Dpos, index=countries, columns=countries).to_csv(Dpos_path)
    pd.DataFrame(Ddir, index=countries, columns=countries).to_csv(Ddir_path)
    pd.DataFrame(Dcombo, index=countries, columns=countries).to_csv(Dcmb_path)
    pd.DataFrame(Opos, index=countries, columns=countries).to_csv(Opos_path)
    pd.DataFrame(Odir, index=countries, columns=countries).to_csv(Odir_path)

    print(f"[{flow}] wrote distance + overlap matrices:")
    print(f"  {Dpos_path}")
    print(f"  {Ddir_path}")
    print(f"  {Dcmb_path}")
    print(f"  {Opos_path}")
    print(f"  {Odir_path}")
    print(f"[{flow}] normalization medians: med_pos={med_pos:.6g}, med_dir={med_dir:.6g}")

    # Cluster using combined distance
    run_hierarchical_clustering(Dcombo, countries, flow, out_prefix)

print("\nDone.")
print("Next: inspect *_silhouette.csv and *_clusters.csv, then we can compute cluster attractor prototypes.")


[exports] wrote distance + overlap matrices:
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Dpos.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Ddir.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Dcombo.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_overlap_years.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_overlap_steps.csv
[exports] normalization medians: med_pos=0.674005, med_dir=0.983593
[exports] selected k=3 silhouette=0.7483
  wrote: C:\Python\trade\geo\gravitationalcluster\gc_exports_silhouette.csv
  wrote: C:\Python\trade\geo\gravitationalcluster\gc_exports_clusters.csv
[imports] wrote distance + overlap matrices:
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Dpos.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Ddir.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Dcombo.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_overlap_years.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports

**For each cluster, compute a fixed “attractor” as a prototype (endpoint-based or direction-based).**

In [28]:
import os
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

STATE_LONG = os.path.join(BASE_DIR, "gc_state_long.csv")

CLUSTERS_EXPORTS = os.path.join(BASE_DIR, "gc_exports_clusters.csv")
CLUSTERS_IMPORTS = os.path.join(BASE_DIR, "gc_imports_clusters.csv")

OUT_EXPORTS = os.path.join(BASE_DIR, "gc_exports_attractors.csv")
OUT_IMPORTS = os.path.join(BASE_DIR, "gc_imports_attractors.csv")
OUT_ALL     = os.path.join(BASE_DIR, "gc_attractors_all.csv")


# ============================================================
# SPHERE HELPERS
# ============================================================
def unit_normalize(v):
    n = np.linalg.norm(v)
    if not np.isfinite(n) or n == 0:
        return np.array([np.nan, np.nan, np.nan])
    return v / n

def spherical_mean_unitvec(xyz):
    """
    xyz: (n,3) unit vectors, possibly with NaNs
    Returns unit vector spherical mean.
    """
    xyz = np.asarray(xyz, dtype=float)
    xyz = xyz[~np.isnan(xyz).any(axis=1)]
    if xyz.shape[0] == 0:
        return np.array([np.nan, np.nan, np.nan])
    v = xyz.sum(axis=0)
    return unit_normalize(v)

def unitvec_to_latlon_deg(xyz):
    """
    xyz unit vector -> (lat, lon) degrees.
    """
    x, y, z = xyz
    if not np.isfinite([x, y, z]).all():
        return np.nan, np.nan
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    # normalize lon to [-180, 180]
    if lon > 180:
        lon -= 360
    if lon < -180:
        lon += 360
    return lat, lon

def chord_direction_unit(x1, x2):
    """
    Unit direction from x1 -> x2 in 3D chord space.
    """
    v = x2 - x1
    return unit_normalize(v)


# ============================================================
# CORE LOGIC
# ============================================================
def compute_attractors_for_flow(state, clusters_df, flow_label):
    """
    Computes endpoint-based and direction-based prototypes by cluster.

    Endpoint-based attractor:
      Spherical mean of each country's endpoint vector.

    Direction-based prototype:
      Mean of each country's last-step direction (chord direction).
      Also expressed as a unit vector and converted to lat/lon for convenience.

    Returns: DataFrame with one row per cluster.
    """
    # restrict to flow
    s = state[state["flow"] == flow_label].copy()
    if s.empty:
        raise ValueError(f"No rows in state_long for flow={flow_label}")

    # keep only countries in cluster file
    cset = set(clusters_df["country"].astype(str))
    s = s[s["country"].astype(str).isin(cset)].copy()

    # Build endpoint table per country: last observed year
    s["year"] = s["year"].astype(int)
    s = s.sort_values(["country", "year"])

    # endpoint rows
    idx_last = s.groupby("country")["year"].idxmax()
    end = s.loc[idx_last, ["country", "year", "lat", "lon", "x", "y", "z"]].copy()
    end = end.rename(columns={"year": "end_year", "lat": "end_lat", "lon": "end_lon",
                              "x": "end_x", "y": "end_y", "z": "end_z"})

    # last-step direction per country: need second-to-last and last
    last_dirs = []
    for country, g in s.groupby("country"):
        g = g.sort_values("year")
        if len(g) < 2:
            last_dirs.append({"country": country, "dir_x": np.nan, "dir_y": np.nan, "dir_z": np.nan, "dir_dt": np.nan})
            continue
        r1 = g.iloc[-2]
        r2 = g.iloc[-1]
        u = chord_direction_unit(
            np.array([r1["x"], r1["y"], r1["z"]], dtype=float),
            np.array([r2["x"], r2["y"], r2["z"]], dtype=float),
        )
        last_dirs.append({
            "country": country,
            "dir_x": u[0], "dir_y": u[1], "dir_z": u[2],
            "dir_dt": int(r2["year"]) - int(r1["year"])
        })
    dirs = pd.DataFrame(last_dirs)

    # Merge endpoints + dirs + clusters
    cl = clusters_df[["country", "cluster"]].copy()
    m = cl.merge(end, on="country", how="left").merge(dirs, on="country", how="left")

    # Compute prototypes per cluster
    out_rows = []
    for k, gk in m.groupby("cluster"):
        # endpoint attractor
        endpoint_xyz = gk[["end_x", "end_y", "end_z"]].to_numpy(dtype=float)
        star_xyz = spherical_mean_unitvec(endpoint_xyz)
        star_lat, star_lon = unitvec_to_latlon_deg(star_xyz)

        # direction prototype (mean of last-step directions)
        dir_xyz = gk[["dir_x", "dir_y", "dir_z"]].to_numpy(dtype=float)
        dir_mean = spherical_mean_unitvec(dir_xyz)
        dir_lat, dir_lon = unitvec_to_latlon_deg(dir_mean)  # interpret as "direction point" only

        # diagnostics
        n_countries = int(gk["country"].nunique())
        n_endpoints = int(np.isfinite(gk["end_x"]).sum())
        n_dirs = int(np.isfinite(gk["dir_x"]).sum())
        end_year_min = int(gk["end_year"].min()) if np.isfinite(gk["end_year"]).any() else np.nan
        end_year_max = int(gk["end_year"].max()) if np.isfinite(gk["end_year"]).any() else np.nan

        out_rows.append({
            "flow": flow_label,
            "cluster": int(k) if np.isfinite(k) else k,

            # Endpoint-based attractor ("fixed star" location)
            "attractor_type": "endpoint",
            "star_lat": star_lat,
            "star_lon": star_lon,
            "star_x": star_xyz[0],
            "star_y": star_xyz[1],
            "star_z": star_xyz[2],

            # Direction-based prototype (stored too)
            "dir_lat": dir_lat,
            "dir_lon": dir_lon,
            "dir_x": dir_mean[0],
            "dir_y": dir_mean[1],
            "dir_z": dir_mean[2],

            # Diagnostics
            "n_countries": n_countries,
            "n_endpoints_used": n_endpoints,
            "n_lastdirs_used": n_dirs,
            "end_year_min": end_year_min,
            "end_year_max": end_year_max,
        })

    return pd.DataFrame(out_rows).sort_values(["flow", "cluster"])


# ============================================================
# RUN
# ============================================================
# Load state long
state = pd.read_csv(STATE_LONG)

required = {"country", "flow", "year", "lat", "lon", "x", "y", "z"}
missing = required - set(state.columns)
if missing:
    raise ValueError(f"gc_state_long.csv missing columns: {sorted(missing)}")

# Load clusters
exp_cl = pd.read_csv(CLUSTERS_EXPORTS)
imp_cl = pd.read_csv(CLUSTERS_IMPORTS)

# Basic checks
for dfc, name in [(exp_cl, "exports"), (imp_cl, "imports")]:
    if not {"country", "flow", "cluster"}.issubset(dfc.columns):
        raise ValueError(f"{name} cluster file missing required columns. Need: country, flow, cluster")
    # allow file to contain flow column, but we will enforce flow label below

# Enforce correct flow label filtering
exp_cl = exp_cl[exp_cl["flow"] == "exports"].copy()
imp_cl = imp_cl[imp_cl["flow"] == "imports"].copy()

# Compute attractors
exports_attr = compute_attractors_for_flow(state, exp_cl, "exports")
imports_attr = compute_attractors_for_flow(state, imp_cl, "imports")

# Save
exports_attr.to_csv(OUT_EXPORTS, index=False)
imports_attr.to_csv(OUT_IMPORTS, index=False)

all_attr = pd.concat([exports_attr, imports_attr], ignore_index=True)
all_attr.to_csv(OUT_ALL, index=False)

print("✓ Wrote attractor prototypes:")
print(f"  {OUT_EXPORTS}")
print(f"  {OUT_IMPORTS}")
print(f"  {OUT_ALL}")

print("\nInterpretation reminder:")
print("  - star_* columns = endpoint-based cluster location (true 'fixed attractor' prototype).")
print("  - dir_* columns  = dominant last-step direction (a vector; lat/lon is only for visualization).")


✓ Wrote attractor prototypes:
  C:\Python\trade\geo\gravitationalcluster\gc_exports_attractors.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_attractors.csv
  C:\Python\trade\geo\gravitationalcluster\gc_attractors_all.csv

Interpretation reminder:
  - star_* columns = endpoint-based cluster location (true 'fixed attractor' prototype).
  - dir_* columns  = dominant last-step direction (a vector; lat/lon is only for visualization).


**Validate stability (cluster robustness over time windows / bootstrap years) and finalize.**

In [29]:
import os
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

STATE_LONG = os.path.join(BASE_DIR, "gc_state_long.csv")
CLUST_EXPORTS = os.path.join(BASE_DIR, "gc_exports_clusters.csv")
CLUST_IMPORTS = os.path.join(BASE_DIR, "gc_imports_clusters.csv")

OUT_SUMMARY = os.path.join(BASE_DIR, "gc_stability_summary.csv")
OUT_BOOT = os.path.join(BASE_DIR, "gc_stability_bootstrap_detail.csv")
OUT_WIN = os.path.join(BASE_DIR, "gc_stability_window_detail.csv")

FLOWS = ["exports", "imports"]

# Distance settings (should match what you used)
MIN_OVERLAP_YEARS = 10
MIN_OVERLAP_STEPS = 8
ALPHA_POS = 0.7
LINKAGE = "average"

# Stability experiments
WINDOWS = [
    ("early_1977_1988", 1977, 1988),
    ("mid_1989_2009", 1989, 2009),
    ("late_2010_2022", 2010, 2022),
]

BOOTSTRAP_REPS = 50
BOOTSTRAP_YEAR_FRACTION = 0.80   # sample ~80% of years with replacement

RANDOM_SEED = 123
rng = np.random.default_rng(RANDOM_SEED)

# ============================================================
# HELPERS (distance construction)
# ============================================================
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def robust_normalize(D):
    finite = np.isfinite(D)
    off = finite & (~np.eye(D.shape[0], dtype=bool))
    vals = D[off]
    if vals.size == 0:
        return D, np.nan
    med = np.median(vals)
    if not np.isfinite(med) or med <= 0:
        return D, med
    return D / med, med

def fill_missing_with_penalty(D, factor=1.5):
    D2 = D.copy()
    finite = np.isfinite(D2)
    maxv = np.nanmax(D2[finite])
    penalty = factor * maxv if maxv > 0 else 1.0
    D2[~finite] = penalty
    np.fill_diagonal(D2, 0.0)
    return D2, penalty

def build_country_year_arrays(df_flow, years_keep):
    """
    Returns:
      countries list
      years np.array sorted (subset)
      X (N,T,3) with NaNs
      M (N,T) mask
    """
    df = df_flow[df_flow["year"].isin(years_keep)].copy()
    years = np.sort(df["year"].unique())
    countries = np.sort(df["country"].unique())

    year_to_idx = {y:i for i,y in enumerate(years)}
    country_to_idx = {c:i for i,c in enumerate(countries)}

    N = len(countries)
    T = len(years)

    X = np.full((N, T, 3), np.nan, dtype=float)
    M = np.zeros((N, T), dtype=bool)

    for r in df.itertuples(index=False):
        i = country_to_idx[r.country]
        t = year_to_idx[r.year]
        X[i, t, 0] = r.x
        X[i, t, 1] = r.y
        X[i, t, 2] = r.z
        M[i, t] = True

    return countries.tolist(), years, X, M

def build_step_unit_vectors(X, M):
    """
    U at index t: unit direction from t -> next available index for that country
    MU mask where defined
    """
    N, T, _ = X.shape
    U = np.full((N, T, 3), np.nan, dtype=float)
    MU = np.zeros((N, T), dtype=bool)

    for i in range(N):
        idx = np.where(M[i])[0]
        if idx.size < 2:
            continue
        for k in range(idx.size - 1):
            t0 = idx[k]
            t1 = idx[k + 1]
            v = X[i, t1] - X[i, t0]
            n = np.linalg.norm(v)
            if not np.isfinite(n) or n == 0:
                continue
            U[i, t0] = v / n
            MU[i, t0] = True
    return U, MU

def pairwise_pos_distance(X, M, min_overlap):
    N, T, _ = X.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    for i in range(N):
        Mi = M[i]
        Xi = X[i]
        for j in range(i+1, N):
            mask = Mi & M[j]
            c = int(mask.sum())
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Xi[mask], X[j, mask])
            ang = arccos_clip(dots)
            D[i, j] = D[j, i] = float(np.mean(ang))
    return D

def pairwise_dir_distance(U, MU, min_overlap):
    N, T, _ = U.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    for i in range(N):
        MUi = MU[i]
        Ui = U[i]
        for j in range(i+1, N):
            mask = MUi & MU[j]
            c = int(mask.sum())
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Ui[mask], U[j, mask])
            mismatch = 1.0 - dots
            D[i, j] = D[j, i] = float(np.mean(mismatch))
    return D

def compute_distance_combo(df_flow, years_keep):
    countries, years, X, M = build_country_year_arrays(df_flow, years_keep)
    if len(countries) < 2:
        raise ValueError("Not enough countries in this subset to compute distances.")

    U, MU = build_step_unit_vectors(X, M)

    Dpos = pairwise_pos_distance(X, M, MIN_OVERLAP_YEARS)
    Ddir = pairwise_dir_distance(U, MU, MIN_OVERLAP_STEPS)

    Dpos_n, _ = robust_normalize(Dpos)
    Ddir_n, _ = robust_normalize(Ddir)

    Dcombo = ALPHA_POS * Dpos_n + (1.0 - ALPHA_POS) * Ddir_n
    Dfilled, penalty = fill_missing_with_penalty(Dcombo, factor=1.5)
    return countries, Dfilled, penalty

def cluster_precomputed(Dfilled, k):
    model = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage=LINKAGE)
    return model.fit_predict(Dfilled)

def align_to_full(countries_sub, labels_sub, full_map):
    """
    Returns series aligned to full countries (NaN for missing countries),
    so ARI/NMI computed on intersection only.
    """
    sub = pd.Series(labels_sub, index=countries_sub, dtype=int)
    full = pd.Series(full_map, dtype=int)

    common = full.index.intersection(sub.index)
    return full.loc[common].to_numpy(), sub.loc[common].to_numpy(), common.size

# ============================================================
# LOAD DATA
# ============================================================
state = pd.read_csv(STATE_LONG)
state["year"] = state["year"].astype(int)

# full-sample cluster labels (as baseline)
full_labels = {}
k_selected = {}

for flow, fp in [("exports", CLUST_EXPORTS), ("imports", CLUST_IMPORTS)]:
    cl = pd.read_csv(fp)
    cl = cl[cl["flow"] == flow].copy()
    # k_selected stored in file as constant column
    k = int(cl["k_selected"].iloc[0])
    k_selected[flow] = k
    full_labels[flow] = cl.set_index("country")["cluster"].astype(int)

# ============================================================
# RUN WINDOW ROBUSTNESS
# ============================================================
win_rows = []

for flow in FLOWS:
    df_flow = state[state["flow"] == flow][["country","year","x","y","z"]].copy()
    years_all = np.sort(df_flow["year"].unique())
    k = k_selected[flow]

    for wname, a, b in WINDOWS:
        years_keep = years_all[(years_all >= a) & (years_all <= b)]
        if years_keep.size < 5:
            continue

        countries_sub, Dfilled, penalty = compute_distance_combo(df_flow, years_keep)
        labels_sub = cluster_precomputed(Dfilled, k)

        full_vec, sub_vec, n_common = align_to_full(countries_sub, labels_sub, full_labels[flow])

        if n_common < 2:
            ari = np.nan
            nmi = np.nan
        else:
            ari = adjusted_rand_score(full_vec, sub_vec)
            nmi = normalized_mutual_info_score(full_vec, sub_vec)

        win_rows.append({
            "flow": flow,
            "test": "window",
            "window": wname,
            "year_min": int(years_keep.min()),
            "year_max": int(years_keep.max()),
            "k": k,
            "n_countries_subset": len(countries_sub),
            "n_countries_common": n_common,
            "missing_penalty_used": penalty,
            "ARI_vs_full": ari,
            "NMI_vs_full": nmi,
        })

win_detail = pd.DataFrame(win_rows).sort_values(["flow","window"])
win_detail.to_csv(OUT_WIN, index=False)

# ============================================================
# RUN BOOTSTRAP YEARS ROBUSTNESS
# ============================================================
boot_rows = []

for flow in FLOWS:
    df_flow = state[state["flow"] == flow][["country","year","x","y","z"]].copy()
    years_all = np.sort(df_flow["year"].unique())
    k = k_selected[flow]

    n_draw = max(5, int(np.ceil(BOOTSTRAP_YEAR_FRACTION * years_all.size)))

    for r in range(BOOTSTRAP_REPS):
        # sample years with replacement, then unique+sort (keeps multiplicity irrelevant for our averaging)
        draw = rng.choice(years_all, size=n_draw, replace=True)
        years_keep = np.sort(np.unique(draw))
        if years_keep.size < 5:
            continue

        countries_sub, Dfilled, penalty = compute_distance_combo(df_flow, years_keep)
        labels_sub = cluster_precomputed(Dfilled, k)

        full_vec, sub_vec, n_common = align_to_full(countries_sub, labels_sub, full_labels[flow])

        if n_common < 2:
            ari = np.nan
            nmi = np.nan
        else:
            ari = adjusted_rand_score(full_vec, sub_vec)
            nmi = normalized_mutual_info_score(full_vec, sub_vec)

        boot_rows.append({
            "flow": flow,
            "test": "bootstrap_years",
            "rep": r,
            "n_years_used": int(years_keep.size),
            "year_min": int(years_keep.min()),
            "year_max": int(years_keep.max()),
            "k": k,
            "n_countries_subset": len(countries_sub),
            "n_countries_common": n_common,
            "missing_penalty_used": penalty,
            "ARI_vs_full": ari,
            "NMI_vs_full": nmi,
        })

boot_detail = pd.DataFrame(boot_rows).sort_values(["flow","rep"])
boot_detail.to_csv(OUT_BOOT, index=False)

# ============================================================
# SUMMARY
# ============================================================
def summarize(df, label):
    g = df.groupby("flow")[["ARI_vs_full","NMI_vs_full"]].agg(["mean","std","min","max"])
    g.columns = [f"{label}_{a}_{b}" for a,b in g.columns]
    return g.reset_index()

sum_win = summarize(win_detail, "window")
sum_boot = summarize(boot_detail, "bootstrap")

summary = sum_win.merge(sum_boot, on="flow", how="outer")
summary.to_csv(OUT_SUMMARY, index=False)

print("✓ Wrote stability outputs:")
print(f"  {OUT_WIN}")
print(f"  {OUT_BOOT}")
print(f"  {OUT_SUMMARY}")

print("\nInterpretation guide:")
print("  - ARI/NMI close to 1 => very stable vs full-sample clustering.")
print("  - If early/mid/late windows disagree, your clusters are regime-dependent (not a failure; it’s a result).")
print("  - If bootstrap variance is high, clusters are sensitive to specific years (consider smoothing or restricting to a stable subperiod).")


✓ Wrote stability outputs:
  C:\Python\trade\geo\gravitationalcluster\gc_stability_window_detail.csv
  C:\Python\trade\geo\gravitationalcluster\gc_stability_bootstrap_detail.csv
  C:\Python\trade\geo\gravitationalcluster\gc_stability_summary.csv

Interpretation guide:
  - ARI/NMI close to 1 => very stable vs full-sample clustering.
  - If early/mid/late windows disagree, your clusters are regime-dependent (not a failure; it’s a result).
  - If bootstrap variance is high, clusters are sensitive to specific years (consider smoothing or restricting to a stable subperiod).


**Atractors**

In [30]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
GEO_DIR = r"C:\Python\trade\geo"
SHAPEFILE = os.path.join(GEO_DIR, r"ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp")

BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
EXPORTS_ATTR = os.path.join(BASE_DIR, "gc_exports_attractors.csv")
IMPORTS_ATTR = os.path.join(BASE_DIR, "gc_imports_attractors.csv")

OUT_PNG = os.path.join(BASE_DIR, "gc_stars_mercator.png")
OUT_PDF = os.path.join(BASE_DIR, "gc_stars_mercator.pdf")

# ------------------------------------------------------------
# LOAD BASEMAP (remove Antarctica)
# ------------------------------------------------------------
world = gpd.read_file(SHAPEFILE)
if "ISO_A3" in world.columns:
    world = world[world["ISO_A3"] != "ATA"].copy()
if world.crs is None:
    world = world.set_crs(epsg=4326)

# ------------------------------------------------------------
# LOAD ATTRACTORS
# ------------------------------------------------------------
exp = pd.read_csv(EXPORTS_ATTR)
imp = pd.read_csv(IMPORTS_ATTR)

# We want the endpoint “stars” (fixed attractors)
# Columns expected: star_lat, star_lon, n_countries
for df, name in [(exp, "exports"), (imp, "imports")]:
    needed = {"star_lat", "star_lon", "n_countries"}
    miss = needed - set(df.columns)
    if miss:
        raise ValueError(f"{name} attractors missing columns: {sorted(miss)}")

# Build GeoDataFrames in WGS84
gexp = gpd.GeoDataFrame(
    exp.copy(),
    geometry=[Point(xy) for xy in zip(exp["star_lon"], exp["star_lat"])],
    crs="EPSG:4326"
)

gimp = gpd.GeoDataFrame(
    imp.copy(),
    geometry=[Point(xy) for xy in zip(imp["star_lon"], imp["star_lat"])],
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# PROJECT TO MERCATOR (EPSG:3857)
# ------------------------------------------------------------
world_m = world.to_crs(epsg=3857)
gexp_m = gexp.to_crs(epsg=3857)
gimp_m = gimp.to_crs(epsg=3857)

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 7))

# Basemap in gray
world_m.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.35, zorder=1)

# Stars (exports=red, imports=blue)
gexp_m.plot(ax=ax, color="red", markersize=40, zorder=3)
gimp_m.plot(ax=ax, color="dodgerblue", markersize=40, zorder=3)

# Label number of countries attracted (n_countries)
# Offset in meters so labels don't sit exactly on dots
dx, dy = 70000, 70000

for _, r in gexp_m.iterrows():
    if pd.notna(r.geometry.x) and pd.notna(r["n_countries"]):
        ax.text(
            r.geometry.x + dx,
            r.geometry.y + dy,
            str(int(r["n_countries"])),
            fontsize=8,
            color="black",
            zorder=4
        )

for _, r in gimp_m.iterrows():
    if pd.notna(r.geometry.x) and pd.notna(r["n_countries"]):
        ax.text(
            r.geometry.x + dx,
            r.geometry.y - dy,
            str(int(r["n_countries"])),
            fontsize=8,
            color="black",
            zorder=4
        )

# Clean look
ax.set_axis_off()
plt.tight_layout()

# Save
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print("✓ Saved:")
print(f"  {OUT_PNG}")
print(f"  {OUT_PDF}")


✓ Saved:
  C:\Python\trade\geo\gravitationalcluster\gc_stars_mercator.png
  C:\Python\trade\geo\gravitationalcluster\gc_stars_mercator.pdf


**Moving Stars** Clusters

In [31]:
import os
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

STATE_LONG = os.path.join(BASE_DIR, "gc_state_long.csv")

OUT_STATE_DEMEAN = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

FLOWS = ["exports", "imports"]

# Distance construction (same as before)
MIN_OVERLAP_YEARS = 10
MIN_OVERLAP_STEPS = 8
ALPHA_POS = 0.7
LINKAGE = "average"
K_MIN, K_MAX = 2, 12

# Output prefixes
PREFIX = "gc"
SUFFIX = "demeaned"

# ============================================================
# HELPERS
# ============================================================
def unit_normalize(v):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    n = np.where((~np.isfinite(n)) | (n == 0), np.nan, n)
    return v / n

def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def robust_normalize(D):
    finite = np.isfinite(D)
    off = finite & (~np.eye(D.shape[0], dtype=bool))
    vals = D[off]
    if vals.size == 0:
        return D, np.nan
    med = np.median(vals)
    if not np.isfinite(med) or med <= 0:
        return D, med
    return D / med, med

def fill_missing_with_penalty(D, factor=1.5):
    D2 = D.copy()
    finite = np.isfinite(D2)
    if not finite.any():
        raise ValueError("Distance matrix has no finite entries.")
    maxv = np.nanmax(D2[finite])
    penalty = factor * maxv if maxv > 0 else 1.0
    D2[~finite] = penalty
    np.fill_diagonal(D2, 0.0)
    return D2, penalty

def build_country_year_arrays(df_flow):
    years = np.sort(df_flow["year"].unique())
    countries = np.sort(df_flow["country"].unique())
    year_to_idx = {y:i for i,y in enumerate(years)}
    country_to_idx = {c:i for i,c in enumerate(countries)}

    N = len(countries)
    T = len(years)

    X = np.full((N, T, 3), np.nan, dtype=float)
    M = np.zeros((N, T), dtype=bool)

    for r in df_flow.itertuples(index=False):
        i = country_to_idx[r.country]
        t = year_to_idx[r.year]
        X[i, t, 0] = r.x
        X[i, t, 1] = r.y
        X[i, t, 2] = r.z
        M[i, t] = True

    return countries.tolist(), years, X, M

def build_step_unit_vectors(X, M):
    N, T, _ = X.shape
    U = np.full((N, T, 3), np.nan, dtype=float)
    MU = np.zeros((N, T), dtype=bool)

    for i in range(N):
        idx = np.where(M[i])[0]
        if idx.size < 2:
            continue
        for k in range(idx.size - 1):
            t0 = idx[k]
            t1 = idx[k + 1]
            v = X[i, t1] - X[i, t0]
            n = np.linalg.norm(v)
            if not np.isfinite(n) or n == 0:
                continue
            U[i, t0] = v / n
            MU[i, t0] = True
    return U, MU

def pairwise_pos_distance(X, M, min_overlap):
    N, T, _ = X.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    Over = np.zeros((N, N), dtype=int)

    for i in range(N):
        Mi = M[i]
        Xi = X[i]
        for j in range(i+1, N):
            mask = Mi & M[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Xi[mask], X[j, mask])
            ang = arccos_clip(dots)
            D[i, j] = D[j, i] = float(np.mean(ang))
    return D, Over

def pairwise_dir_distance(U, MU, min_overlap):
    N, T, _ = U.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    Over = np.zeros((N, N), dtype=int)

    for i in range(N):
        MUi = MU[i]
        Ui = U[i]
        for j in range(i+1, N):
            mask = MUi & MU[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Ui[mask], U[j, mask])
            mismatch = 1.0 - dots
            D[i, j] = D[j, i] = float(np.mean(mismatch))
    return D, Over

def choose_k_and_cluster(Dcombo, countries, flow):
    Dfilled, penalty = fill_missing_with_penalty(Dcombo, factor=1.5)

    sil_rows = []
    best = {"k": None, "sil": -np.inf, "labels": None}

    for k in range(K_MIN, K_MAX + 1):
        model = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage=LINKAGE
        )
        labels = model.fit_predict(Dfilled)
        sil = silhouette_score(Dfilled, labels, metric="precomputed")
        sil_rows.append({"flow": flow, "k": k, "silhouette": sil})

        if sil > best["sil"]:
            best = {"k": k, "sil": sil, "labels": labels}

    sil_df = pd.DataFrame(sil_rows)
    labels_df = pd.DataFrame({
        "country": countries,
        "flow": flow,
        "cluster": best["labels"],
        "k_selected": best["k"],
        "silhouette_selected": best["sil"],
        "missing_penalty_used": penalty,
        "linkage": LINKAGE,
        "alpha_pos": ALPHA_POS,
        "min_overlap_years": MIN_OVERLAP_YEARS,
        "min_overlap_steps": MIN_OVERLAP_STEPS,
        "demeaned": True
    }).sort_values(["cluster", "country"])

    return labels_df, sil_df

# ============================================================
# STEP 1: BUILD CO-MOVING (DEMEANED) STATE FILE
# ============================================================
state = pd.read_csv(STATE_LONG)
need = {"country","flow","year","x","y","z"}
missing = need - set(state.columns)
if missing:
    raise ValueError(f"{STATE_LONG} missing columns: {sorted(missing)}")

state["year"] = state["year"].astype(int)

# Compute global spherical mean per (flow, year), then subtract+renormalize
demeaned_parts = []

for flow in FLOWS:
    sf = state[state["flow"] == flow].copy()
    if sf.empty:
        continue

    # global mean vector per year
    g = (
        sf.groupby("year")[["x","y","z"]]
        .sum()
        .reset_index()
    )
    gv = g[["x","y","z"]].to_numpy(dtype=float)
    gv = unit_normalize(gv)  # (T,3)
    g["gx"], g["gy"], g["gz"] = gv[:,0], gv[:,1], gv[:,2]
    g = g[["year","gx","gy","gz"]]

    # merge and demean
    sf = sf.merge(g, on="year", how="left")

    r = sf[["x","y","z"]].to_numpy(dtype=float) - sf[["gx","gy","gz"]].to_numpy(dtype=float)
    r = unit_normalize(r)

    sf["x_demean"] = r[:,0]
    sf["y_demean"] = r[:,1]
    sf["z_demean"] = r[:,2]

    demeaned_parts.append(sf)

state_dm = pd.concat(demeaned_parts, ignore_index=True)

# Write a compact version (keep original x/y/z + demeaned + global mean)
cols_keep = ["country","flow","year","lat","lon","x","y","z","weight","gx","gy","gz","x_demean","y_demean","z_demean"]
cols_keep = [c for c in cols_keep if c in state_dm.columns]
state_dm[cols_keep].to_csv(OUT_STATE_DEMEAN, index=False)
print(f"✓ wrote: {OUT_STATE_DEMEAN}")

# ============================================================
# STEP 2: RECOMPUTE DISTANCES + CLUSTERS USING DEMEANED VECTORS
# ============================================================
for flow in FLOWS:
    df_flow = state_dm[state_dm["flow"] == flow][["country","year","x_demean","y_demean","z_demean"]].copy()
    if df_flow.empty:
        print(f"Skipping {flow}: no demeaned data.")
        continue

    df_flow = df_flow.rename(columns={"x_demean":"x","y_demean":"y","z_demean":"z"})

    # Build arrays
    countries, years, X, M = build_country_year_arrays(df_flow)

    # Step directions
    U, MU = build_step_unit_vectors(X, M)

    # Distances
    Dpos, Opos = pairwise_pos_distance(X, M, MIN_OVERLAP_YEARS)
    Ddir, Odir = pairwise_dir_distance(U, MU, MIN_OVERLAP_STEPS)

    # Normalize + combine
    Dpos_n, med_pos = robust_normalize(Dpos)
    Ddir_n, med_dir = robust_normalize(Ddir)
    Dcombo = ALPHA_POS * Dpos_n + (1.0 - ALPHA_POS) * Ddir_n

    # Save matrices
    Dpos_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_Dpos_{SUFFIX}.csv")
    Ddir_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_Ddir_{SUFFIX}.csv")
    Dcmb_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_Dcombo_{SUFFIX}.csv")
    Opos_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_overlap_years_{SUFFIX}.csv")
    Odir_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_overlap_steps_{SUFFIX}.csv")

    pd.DataFrame(Dpos, index=countries, columns=countries).to_csv(Dpos_path)
    pd.DataFrame(Ddir, index=countries, columns=countries).to_csv(Ddir_path)
    pd.DataFrame(Dcombo, index=countries, columns=countries).to_csv(Dcmb_path)
    pd.DataFrame(Opos, index=countries, columns=countries).to_csv(Opos_path)
    pd.DataFrame(Odir, index=countries, columns=countries).to_csv(Odir_path)

    print(f"[{flow}] wrote demeaned distances:")
    print(f"  {Dpos_path}")
    print(f"  {Ddir_path}")
    print(f"  {Dcmb_path}")
    print(f"  med_pos={med_pos:.6g}, med_dir={med_dir:.6g}")

    # Cluster (choose k by silhouette)
    labels_df, sil_df = choose_k_and_cluster(Dcombo, countries, flow)

    clusters_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_clusters_{SUFFIX}.csv")
    sil_path = os.path.join(BASE_DIR, f"{PREFIX}_{flow}_silhouette_{SUFFIX}.csv")

    labels_df.to_csv(clusters_path, index=False)
    sil_df.to_csv(sil_path, index=False)

    print(f"[{flow}] wrote:")
    print(f"  {clusters_path}")
    print(f"  {sil_path}")

print("\nDone. Next step: recompute attractor prototypes for these *_clusters_demeaned.csv if needed.")


✓ wrote: C:\Python\trade\geo\gravitationalcluster\gc_state_long_demeaned.csv
[exports] wrote demeaned distances:
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Dpos_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Ddir_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_Dcombo_demeaned.csv
  med_pos=1.49323, med_dir=0.973314
[exports] wrote:
  C:\Python\trade\geo\gravitationalcluster\gc_exports_clusters_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_exports_silhouette_demeaned.csv
[imports] wrote demeaned distances:
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Dpos_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Ddir_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_Dcombo_demeaned.csv
  med_pos=1.49162, med_dir=0.971695
[imports] wrote:
  C:\Python\trade\geo\gravitationalcluster\gc_imports_clusters_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_silhouette_demeaned.csv



In [32]:
import os
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"

STATE_DEMEAN = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

CLUST_EXPORTS_DM = os.path.join(BASE_DIR, "gc_exports_clusters_demeaned.csv")
CLUST_IMPORTS_DM = os.path.join(BASE_DIR, "gc_imports_clusters_demeaned.csv")

OUT_EXPORTS_DM = os.path.join(BASE_DIR, "gc_exports_attractors_demeaned.csv")
OUT_IMPORTS_DM = os.path.join(BASE_DIR, "gc_imports_attractors_demeaned.csv")
OUT_ALL_DM     = os.path.join(BASE_DIR, "gc_attractors_all_demeaned.csv")


# ============================================================
# HELPERS
# ============================================================
def unit_normalize(v):
    n = np.linalg.norm(v)
    if not np.isfinite(n) or n == 0:
        return np.array([np.nan, np.nan, np.nan])
    return v / n

def spherical_mean_unitvec(xyz):
    xyz = np.asarray(xyz, dtype=float)
    xyz = xyz[~np.isnan(xyz).any(axis=1)]
    if xyz.shape[0] == 0:
        return np.array([np.nan, np.nan, np.nan])
    return unit_normalize(xyz.sum(axis=0))

def unitvec_to_latlon_deg(xyz):
    x, y, z = xyz
    if not np.isfinite([x, y, z]).all():
        return np.nan, np.nan
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    # [-180, 180]
    if lon > 180:
        lon -= 360
    if lon < -180:
        lon += 360
    return lat, lon


def compute_attractors_demeaned(state_dm, clusters_df, flow):
    """
    Endpoint-based prototype in demeaned space:
      spherical mean of each country's endpoint demeaned vector.

    Also computes a direction prototype from last-step demeaned direction.
    """
    s = state_dm[state_dm["flow"] == flow].copy()
    cset = set(clusters_df["country"].astype(str))
    s = s[s["country"].astype(str).isin(cset)].copy()

    s["year"] = s["year"].astype(int)
    s = s.sort_values(["country", "year"])

    # endpoint per country (last year)
    idx_last = s.groupby("country")["year"].idxmax()
    end = s.loc[idx_last, ["country", "year", "x_demean", "y_demean", "z_demean"]].copy()
    end = end.rename(columns={"year": "end_year", "x_demean": "end_x", "y_demean": "end_y", "z_demean": "end_z"})

    # last-step direction per country
    last_dirs = []
    for country, g in s.groupby("country"):
        g = g.sort_values("year")
        if len(g) < 2:
            last_dirs.append({"country": country, "dir_x": np.nan, "dir_y": np.nan, "dir_z": np.nan})
            continue
        r1 = g.iloc[-2][["x_demean","y_demean","z_demean"]].to_numpy(dtype=float)
        r2 = g.iloc[-1][["x_demean","y_demean","z_demean"]].to_numpy(dtype=float)
        u = unit_normalize(r2 - r1)
        last_dirs.append({"country": country, "dir_x": u[0], "dir_y": u[1], "dir_z": u[2]})

    dirs = pd.DataFrame(last_dirs)

    # merge with clusters
    m = clusters_df[["country", "cluster"]].merge(end, on="country", how="left").merge(dirs, on="country", how="left")

    out = []
    for k, gk in m.groupby("cluster"):
        endpoint_xyz = gk[["end_x","end_y","end_z"]].to_numpy(dtype=float)
        star_xyz = spherical_mean_unitvec(endpoint_xyz)
        star_lat, star_lon = unitvec_to_latlon_deg(star_xyz)

        dir_xyz = gk[["dir_x","dir_y","dir_z"]].to_numpy(dtype=float)
        dir_mean = spherical_mean_unitvec(dir_xyz)
        dir_lat, dir_lon = unitvec_to_latlon_deg(dir_mean)

        out.append({
            "flow": flow,
            "cluster": int(k),
            # "star" = cluster prototype in demeaned space (interpret as deviation pole)
            "star_lat": star_lat,
            "star_lon": star_lon,
            "star_x": star_xyz[0],
            "star_y": star_xyz[1],
            "star_z": star_xyz[2],
            # direction prototype
            "dir_lat": dir_lat,
            "dir_lon": dir_lon,
            "dir_x": dir_mean[0],
            "dir_y": dir_mean[1],
            "dir_z": dir_mean[2],
            # diagnostics
            "n_countries": int(gk["country"].nunique()),
            "n_endpoints_used": int(np.isfinite(gk["end_x"]).sum()),
            "n_lastdirs_used": int(np.isfinite(gk["dir_x"]).sum()),
            "end_year_min": int(gk["end_year"].min()) if np.isfinite(gk["end_year"]).any() else np.nan,
            "end_year_max": int(gk["end_year"].max()) if np.isfinite(gk["end_year"]).any() else np.nan,
            "demeaned": True
        })

    return pd.DataFrame(out).sort_values(["flow", "cluster"])


# ============================================================
# RUN
# ============================================================
state_dm = pd.read_csv(STATE_DEMEAN)

req = {"country","flow","year","x_demean","y_demean","z_demean"}
missing = req - set(state_dm.columns)
if missing:
    raise ValueError(f"{STATE_DEMEAN} missing columns: {sorted(missing)}")

exp_cl = pd.read_csv(CLUST_EXPORTS_DM)
imp_cl = pd.read_csv(CLUST_IMPORTS_DM)

# filter by flow just to be safe
exp_cl = exp_cl[exp_cl["flow"] == "exports"].copy()
imp_cl = imp_cl[imp_cl["flow"] == "imports"].copy()

exports_attr = compute_attractors_demeaned(state_dm, exp_cl, "exports")
imports_attr = compute_attractors_demeaned(state_dm, imp_cl, "imports")

exports_attr.to_csv(OUT_EXPORTS_DM, index=False)
imports_attr.to_csv(OUT_IMPORTS_DM, index=False)

all_attr = pd.concat([exports_attr, imports_attr], ignore_index=True)
all_attr.to_csv(OUT_ALL_DM, index=False)

print("✓ wrote:")
print(f"  {OUT_EXPORTS_DM}")
print(f"  {OUT_IMPORTS_DM}")
print(f"  {OUT_ALL_DM}")
print("\nNote: these 'stars' are deviation poles in the co-moving frame, not literal Earth locations.")


✓ wrote:
  C:\Python\trade\geo\gravitationalcluster\gc_exports_attractors_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_imports_attractors_demeaned.csv
  C:\Python\trade\geo\gravitationalcluster\gc_attractors_all_demeaned.csv

Note: these 'stars' are deviation poles in the co-moving frame, not literal Earth locations.


**Clusters by two Periods**

In [33]:
# ============================================================
# Gravitational / Directional Period Clustering (READY TO RUN)
# - Splits data into two periods (or more, configurable)
# - Builds two distances:
#     Dpos: average angular separation of positions (requires overlap in years)
#     Ddir: average mismatch in direction of movement (requires overlap in steps)
# - Robustly mixes Dpos + Ddir when Dpos is sufficiently identified
# - Chooses k by silhouette over agglomerative clustering on precomputed distances
# - Writes out distance matrices, overlaps, silhouettes, clusters, and a summary
# ============================================================

import os
import numpy as np
import pandas as pd
import datetime as _dt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
STATE_DEMEAN = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

FLOWS = ["exports", "imports"]

# Periods (inclusive endpoints)
PERIODS = [
    ("1977_2000", 1977, 2000),
    ("2001_2022", 2001, 2022),
]

# Fixed overlap thresholds (as requested)
MIN_OVERLAP_YEARS = 10   # for Dpos
MIN_OVERLAP_STEPS = 8    # for Ddir

# Distance mixing weight (only when Dpos identified)
ALPHA_POS = 0.7

# Clustering
LINKAGE = "average"
K_MIN, K_MAX = 2, 12

# Identification of Dpos: scale threshold with N (recommended)
MIN_FINITE_DPOS_FRACTION = 0.25  # require >= 25% of all directed off-diagonal pairs finite
MIN_FINITE_DPOS_HARD_FLOOR = 50  # but never require less than 50 finite pairs

# Missing-distance penalty multiplier for clustering
MISSING_PENALTY_FACTOR = 1.5

# Run label for outputs
SUFFIX = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")

# ============================================================
# HELPERS
# ============================================================
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def robust_normalize(D):
    """Divide by median finite off-diagonal to put distance matrices on comparable scale."""
    finite = np.isfinite(D)
    off = finite & (~np.eye(D.shape[0], dtype=bool))
    vals = D[off]
    if vals.size == 0:
        return D, np.nan
    med = np.median(vals)
    if not np.isfinite(med) or med <= 0:
        return D, med
    return D / med, med

def fill_missing_with_penalty(D, factor=MISSING_PENALTY_FACTOR):
    """Replace NaNs with 'far' penalty distance so clustering can run."""
    D2 = D.copy()
    finite = np.isfinite(D2)
    if not finite.any():
        raise ValueError("Distance matrix has no finite entries.")
    maxv = np.nanmax(D2[finite])
    penalty = factor * maxv if maxv > 0 else 1.0
    D2[~finite] = penalty
    np.fill_diagonal(D2, 0.0)
    return D2, penalty

def build_country_year_arrays(df_flow):
    years = np.sort(df_flow["year"].unique())
    countries = np.sort(df_flow["country"].unique())
    year_to_idx = {y: i for i, y in enumerate(years)}
    country_to_idx = {c: i for i, c in enumerate(countries)}

    N = len(countries)
    T = len(years)

    X = np.full((N, T, 3), np.nan, dtype=float)
    M = np.zeros((N, T), dtype=bool)

    for r in df_flow.itertuples(index=False):
        i = country_to_idx[r.country]
        t = year_to_idx[r.year]
        X[i, t, 0] = r.x
        X[i, t, 1] = r.y
        X[i, t, 2] = r.z
        M[i, t] = True

    return countries.tolist(), years, X, M

def build_step_unit_vectors(X, M):
    """
    For each country i, create unit step vectors between consecutive observed years:
      U[i,t0] = normalize( X[i,t1] - X[i,t0] ) for consecutive observed indices t0->t1
    MU marks where steps exist (stored at t0).
    """
    N, T, _ = X.shape
    U = np.full((N, T, 3), np.nan, dtype=float)
    MU = np.zeros((N, T), dtype=bool)

    for i in range(N):
        idx = np.where(M[i])[0]
        if idx.size < 2:
            continue
        for k in range(idx.size - 1):
            t0 = idx[k]
            t1 = idx[k + 1]
            v = X[i, t1] - X[i, t0]
            n = np.linalg.norm(v)
            if not np.isfinite(n) or n == 0:
                continue
            U[i, t0] = v / n
            MU[i, t0] = True

    return U, MU

def pairwise_pos_distance(X, M, min_overlap):
    """Mean angular separation across overlapping years."""
    N, T, _ = X.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    Over = np.zeros((N, N), dtype=int)

    for i in range(N):
        Mi = M[i]
        Xi = X[i]
        for j in range(i + 1, N):
            mask = Mi & M[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Xi[mask], X[j, mask])
            ang = arccos_clip(dots)
            D[i, j] = D[j, i] = float(np.mean(ang))

    return D, Over

def pairwise_dir_distance(U, MU, min_overlap):
    """Mean (1 - dot) mismatch across overlapping steps."""
    N, T, _ = U.shape
    D = np.full((N, N), np.nan, dtype=float)
    np.fill_diagonal(D, 0.0)
    Over = np.zeros((N, N), dtype=int)

    for i in range(N):
        MUi = MU[i]
        Ui = U[i]
        for j in range(i + 1, N):
            mask = MUi & MU[j]
            c = int(mask.sum())
            Over[i, j] = Over[j, i] = c
            if c < min_overlap:
                continue
            dots = np.einsum("ij,ij->i", Ui[mask], U[j, mask])
            mismatch = 1.0 - dots
            D[i, j] = D[j, i] = float(np.mean(mismatch))

    return D, Over

def finite_offdiag_count(D):
    finite = np.isfinite(D)
    off = finite & (~np.eye(D.shape[0], dtype=bool))
    return int(off.sum())

def min_pairs_required(N, frac=MIN_FINITE_DPOS_FRACTION, hard_floor=MIN_FINITE_DPOS_HARD_FLOOR):
    # Directed off-diagonals count = N*(N-1)
    return max(hard_floor, int(np.ceil(frac * (N * (N - 1)))))

def mix_distances_fallback(Dpos_n, Ddir_n, alpha=0.7):
    """
    Robust mix without inducing NaNs:
      - both finite: alpha*Dpos + (1-alpha)*Ddir
      - only one finite: use available one
      - neither finite: NaN
    """
    Duse = np.full_like(Dpos_n, np.nan, dtype=float)

    fpos = np.isfinite(Dpos_n)
    fdir = np.isfinite(Ddir_n)

    both = fpos & fdir
    pos_only = fpos & (~fdir)
    dir_only = fdir & (~fpos)

    Duse[both] = alpha * Dpos_n[both] + (1.0 - alpha) * Ddir_n[both]
    Duse[pos_only] = Dpos_n[pos_only]
    Duse[dir_only] = Ddir_n[dir_only]

    np.fill_diagonal(Duse, 0.0)
    return Duse

def choose_k_by_silhouette(D_pre, flow, period_tag):
    """
    Cluster for k in [K_MIN..K_MAX] using agglomerative clustering with precomputed distances.
    Select k with max silhouette (also computed on precomputed distances).
    """
    Dfilled, penalty = fill_missing_with_penalty(D_pre, factor=MISSING_PENALTY_FACTOR)

    # Degeneracy check: if all off-diagonal distances are identical, silhouette isn't meaningful.
    finite = np.isfinite(Dfilled)
    off = finite & (~np.eye(Dfilled.shape[0], dtype=bool))
    vals = Dfilled[off]
    if vals.size == 0 or np.nanstd(vals) == 0:
        N = Dfilled.shape[0]
        labels = np.zeros(N, dtype=int)
        best = {"k": 1 if N == 1 else 2, "sil": np.nan, "labels": labels}
        sil_df = pd.DataFrame([{"flow": flow, "period": period_tag, "k": best["k"], "silhouette": np.nan}])
        return best, sil_df, penalty

    rows = []
    best = {"k": None, "sil": -np.inf, "labels": None}

    for k in range(K_MIN, K_MAX + 1):
        model = AgglomerativeClustering(
            n_clusters=k,
            metric="precomputed",
            linkage=LINKAGE,
        )
        labels = model.fit_predict(Dfilled)
        sil = silhouette_score(Dfilled, labels, metric="precomputed")

        rows.append({"flow": flow, "period": period_tag, "k": k, "silhouette": sil})
        if sil > best["sil"]:
            best = {"k": k, "sil": sil, "labels": labels}

    sil_df = pd.DataFrame(rows)
    return best, sil_df, penalty

# ============================================================
# MAIN
# ============================================================
def main():
    os.makedirs(BASE_DIR, exist_ok=True)

    state = pd.read_csv(STATE_DEMEAN)
    req = {"country", "flow", "year", "x_demean", "y_demean", "z_demean"}
    missing = req - set(state.columns)
    if missing:
        raise ValueError(f"{STATE_DEMEAN} missing columns: {sorted(missing)}")

    state["year"] = state["year"].astype(int)

    summary_rows = []

    for flow in FLOWS:
        sf = state[state["flow"] == flow].copy()
        if sf.empty:
            continue

        # Use demeaned vectors as x/y/z
        sf = sf.rename(columns={"x_demean": "x", "y_demean": "y", "z_demean": "z"})
        sf = sf[["country", "year", "x", "y", "z"]]

        for tag, y0, y1 in PERIODS:
            dfp = sf[(sf["year"] >= y0) & (sf["year"] <= y1)].copy()

            if dfp.empty:
                summary_rows.append({
                    "flow": flow, "period": tag, "year_min": y0, "year_max": y1,
                    "status": "no_data", "alpha_eff": np.nan, "k_selected": np.nan,
                    "silhouette_selected": np.nan, "finite_pairs_Dpos": 0, "finite_pairs_Ddir": 0,
                    "suffix": SUFFIX
                })
                continue

            countries, years, X, M = build_country_year_arrays(dfp)
            U, MU = build_step_unit_vectors(X, M)

            Dpos, Opos = pairwise_pos_distance(X, M, MIN_OVERLAP_YEARS)
            Ddir, Odir = pairwise_dir_distance(U, MU, MIN_OVERLAP_STEPS)

            npos = finite_offdiag_count(Dpos)
            ndir = finite_offdiag_count(Ddir)

            # Normalize components (if possible)
            Dpos_n, med_pos = robust_normalize(Dpos)
            Ddir_n, med_dir = robust_normalize(Ddir)

            # Decide whether Dpos is "identified"
            N = len(countries)
            min_required = min_pairs_required(N)

            if npos < min_required:
                alpha_eff = 0.0
                Duse = Ddir_n.copy()
                np.fill_diagonal(Duse, 0.0)
                status = f"used_dir_only_Dpos_not_identified(npos={npos}<min={min_required})"
            else:
                alpha_eff = ALPHA_POS
                Duse = mix_distances_fallback(Dpos_n, Ddir_n, alpha=alpha_eff)
                status = f"used_combo(alpha={alpha_eff},npos={npos}>=min={min_required})"

            # Save distance + overlap matrices for this period
            base = os.path.join(BASE_DIR, f"gc_{flow}_{tag}_{SUFFIX}")
            pd.DataFrame(Dpos, index=countries, columns=countries).to_csv(base + "_Dpos.csv")
            pd.DataFrame(Ddir, index=countries, columns=countries).to_csv(base + "_Ddir.csv")
            pd.DataFrame(Duse, index=countries, columns=countries).to_csv(base + "_Dused.csv")
            pd.DataFrame(Opos, index=countries, columns=countries).to_csv(base + "_overlap_years.csv")
            pd.DataFrame(Odir, index=countries, columns=countries).to_csv(base + "_overlap_steps.csv")

            # Cluster (k selected by silhouette)
            best, sil_df, penalty = choose_k_by_silhouette(Duse, flow, tag)

            # Save silhouette
            sil_df.to_csv(base + "_silhouette.csv", index=False)

            # Save clusters
            cl_df = pd.DataFrame({
                "country": countries,
                "flow": flow,
                "period": tag,
                "cluster": best["labels"],
                "k_selected": best["k"],
                "silhouette_selected": best["sil"],
                "alpha_eff": alpha_eff,
                "status": status,
                "min_overlap_years": MIN_OVERLAP_YEARS,
                "min_overlap_steps": MIN_OVERLAP_STEPS,
                "linkage": LINKAGE,
                "missing_penalty_used": penalty,
                "med_pos_norm": med_pos,
                "med_dir_norm": med_dir,
                "finite_pairs_Dpos": npos,
                "finite_pairs_Ddir": ndir,
                "suffix": SUFFIX,
            }).sort_values(["cluster", "country"])
            cl_df.to_csv(base + "_clusters.csv", index=False)

            summary_rows.append({
                "flow": flow,
                "period": tag,
                "year_min": y0,
                "year_max": y1,
                "n_countries": len(countries),
                "n_years_in_window": int(len(years)),
                "finite_pairs_Dpos": npos,
                "finite_pairs_Ddir": ndir,
                "alpha_eff": alpha_eff,
                "status": status,
                "k_selected": int(best["k"]) if best["k"] is not None else np.nan,
                "silhouette_selected": float(best["sil"]) if np.isfinite(best["sil"]) else np.nan,
                "min_pairs_required_for_Dpos": min_required,
                "suffix": SUFFIX,
            })

            print(f"[{flow} {tag}] {status}  k={best['k']}  sil={best['sil'] if np.isfinite(best['sil']) else np.nan}")

    summary = pd.DataFrame(summary_rows)
    summary_path = os.path.join(BASE_DIR, f"gc_period_clustering_summary_{SUFFIX}.csv")
    summary.to_csv(summary_path, index=False)

    print("\n✓ Wrote summary:")
    print(f"  {summary_path}")
    print("\nNotes:")
    print(f"  - Thresholds fixed: MIN_OVERLAP_YEARS={MIN_OVERLAP_YEARS}, MIN_OVERLAP_STEPS={MIN_OVERLAP_STEPS}.")
    print(f"  - Dpos identification: >= max({MIN_FINITE_DPOS_HARD_FLOOR}, {MIN_FINITE_DPOS_FRACTION:.2f} * N*(N-1)) finite off-diagonal pairs.")
    print("  - Robust mixing: uses available component when the other is missing (does not create NaNs unnecessarily).")

if __name__ == "__main__":
    main()


[exports 1977_2000] used_combo(alpha=0.7,npos=39362>=min=14702)  k=7  sil=0.6982663348373656
[exports 2001_2022] used_combo(alpha=0.7,npos=55432>=min=14340)  k=5  sil=0.7441364029089723
[imports 1977_2000] used_combo(alpha=0.7,npos=39352>=min=14702)  k=5  sil=0.6934169427146861
[imports 2001_2022] used_combo(alpha=0.7,npos=55432>=min=14102)  k=3  sil=0.760284944620897

✓ Wrote summary:
  C:\Python\trade\geo\gravitationalcluster\gc_period_clustering_summary_20260129_020758.csv

Notes:
  - Thresholds fixed: MIN_OVERLAP_YEARS=10, MIN_OVERLAP_STEPS=8.
  - Dpos identification: >= max(50, 0.25 * N*(N-1)) finite off-diagonal pairs.
  - Robust mixing: uses available component when the other is missing (does not create NaNs unnecessarily).


**Atractors for two periods**

In [34]:
# ============================================================
# Compute "Attractors / Stars" per (flow, period, cluster)
# READY TO RUN
#
# Inputs:
#   - gc_state_long_demeaned.csv  (panel with x_demean,y_demean,z_demean)
#   - gc_{flow}_{period}_{SUFFIX}_clusters.csv  (from your clustering run)
#
# Behavior:
#   - AUTO-DETECTS the most recent clustering SUFFIX in BASE_DIR
#   - For each (flow, period, cluster):
#       * Computes spherical barycenter ("star") from pooled country-year vectors
#       * Outputs star vector + (lat, lon) + dispersion diagnostics
#   - Also outputs per-country diagnostics: mean/median/max distance to star
#
# Outputs:
#   - gc_stars_<SUFFIX>.csv
#   - gc_star_members_<SUFFIX>.csv
# ============================================================

import os
import re
import numpy as np
import pandas as pd

# ============================================================
# CONFIG (match your pipeline)
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
STATE_DEMEAN = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

FLOWS = ["exports", "imports"]
PERIODS = [
    ("1977_2000", 1977, 2000),
    ("2001_2022", 2001, 2022),
]

# Use demeaned vectors for stars (consistent with clustering on demeaned vectors)
USE_DEMEANED_VECTORS = True


# ============================================================
# AUTO-DETECT MOST RECENT CLUSTERING SUFFIX
# ============================================================
def detect_latest_suffix(base_dir):
    """
    Finds the most recent gc_*_*_<SUFFIX>_clusters.csv file in base_dir
    and extracts <SUFFIX> of form YYYYMMDD_HHMMSS.
    """
    pat = re.compile(r"gc_(exports|imports)_(\d{4}_\d{4})_(\d{8}_\d{6})_clusters\.csv")
    suffixes = []
    for fn in os.listdir(base_dir):
        m = pat.match(fn)
        if m:
            suffixes.append(m.group(3))
    if not suffixes:
        raise FileNotFoundError(
            "No clustering files found in BASE_DIR. Expected files like:\n"
            "  gc_exports_1977_2000_YYYYMMDD_HHMMSS_clusters.csv"
        )
    return sorted(suffixes)[-1]  # lexicographic works for timestamp strings

SUFFIX = detect_latest_suffix(BASE_DIR)
print(f"[INFO] Using detected clustering SUFFIX = {SUFFIX}")

OUT_STARS = os.path.join(BASE_DIR, f"gc_stars_{SUFFIX}.csv")
OUT_MEMBERS = os.path.join(BASE_DIR, f"gc_star_members_{SUFFIX}.csv")


# ============================================================
# GEOMETRY HELPERS
# ============================================================
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def unitvec_to_latlon_deg(v):
    """
    v: (3,) unit vector in R^3
    returns (lat_deg, lon_deg) with lon in (-180, 180]
    """
    x, y, z = float(v[0]), float(v[1]), float(v[2])
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    if lon <= -180:
        lon += 360
    elif lon > 180:
        lon -= 360
    return float(lat), float(lon)

def spherical_mean_unitvec(V):
    """
    V: (n,3) array of unit-ish vectors (already filtered finite)
    returns unit vector (3,)
    """
    s = np.sum(V, axis=0)
    n = np.linalg.norm(s)
    if not np.isfinite(n) or n == 0:
        return np.array([np.nan, np.nan, np.nan], dtype=float)
    return s / n

def angular_distance_deg(V, star):
    """
    V: (n,3) unit vectors; star: (3,) unit vector
    returns angular distances in degrees
    """
    dots = np.einsum("ij,j->i", V, star)
    ang = arccos_clip(dots)
    return np.degrees(ang)


# ============================================================
# IO HELPERS
# ============================================================
def clusters_path(flow, period_tag):
    return os.path.join(BASE_DIR, f"gc_{flow}_{period_tag}_{SUFFIX}_clusters.csv")

def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")


# ============================================================
# MAIN
# ============================================================
def main():
    assert_exists(STATE_DEMEAN)

    state = pd.read_csv(STATE_DEMEAN)
    state["year"] = state["year"].astype(int)

    if USE_DEMEANED_VECTORS:
        req = {"country", "flow", "year", "x_demean", "y_demean", "z_demean"}
        missing = req - set(state.columns)
        if missing:
            raise ValueError(f"{STATE_DEMEAN} missing columns: {sorted(missing)}")
        state = state.rename(columns={"x_demean": "x", "y_demean": "y", "z_demean": "z"})
    else:
        req = {"country", "flow", "year", "x", "y", "z"}
        missing = req - set(state.columns)
        if missing:
            raise ValueError(f"{STATE_DEMEAN} missing columns: {sorted(missing)}")

    state = state[["country", "flow", "year", "x", "y", "z"]].copy()

    star_rows = []
    member_rows = []

    for flow in FLOWS:
        sf = state[state["flow"] == flow].copy()
        if sf.empty:
            print(f"[WARN] No rows for flow={flow} in state file.")
            continue

        for period_tag, y0, y1 in PERIODS:
            cl_path = clusters_path(flow, period_tag)
            assert_exists(cl_path)
            cl = pd.read_csv(cl_path)

            reqc = {"country", "cluster"}
            missingc = reqc - set(cl.columns)
            if missingc:
                raise ValueError(f"{cl_path} missing columns: {sorted(missingc)}")

            dfp = sf[(sf["year"] >= y0) & (sf["year"] <= y1)].copy()
            dfm = dfp.merge(cl[["country", "cluster"]], on="country", how="inner")

            if dfm.empty:
                print(f"[WARN] Empty merged data for {flow} {period_tag}.")
                continue

            # drop any non-finite vectors
            ok = np.isfinite(dfm[["x", "y", "z"]]).all(axis=1)
            dfm = dfm[ok].copy()
            if dfm.empty:
                print(f"[WARN] No finite vectors for {flow} {period_tag}.")
                continue

            # Compute stars per cluster
            clusters = sorted(dfm["cluster"].unique())
            for k in clusters:
                gk = dfm[dfm["cluster"] == k].copy()
                if gk.empty:
                    continue

                V = gk[["x", "y", "z"]].to_numpy(dtype=float)

                # normalize rows defensively
                norms = np.linalg.norm(V, axis=1)
                keep = np.isfinite(norms) & (norms > 0)
                V = V[keep]
                if V.shape[0] == 0:
                    continue
                V = V / norms[keep][:, None]

                star = spherical_mean_unitvec(V)
                if not np.isfinite(star).all():
                    continue

                ang_deg = angular_distance_deg(V, star)

                # Member diagnostics (per country)
                gk2 = gk.iloc[np.where(keep)[0]].copy()
                gk2["ang_deg_to_star"] = ang_deg
                per_country = gk2.groupby("country", as_index=False).agg(
                    n_obs=("ang_deg_to_star", "size"),
                    mean_ang_deg=("ang_deg_to_star", "mean"),
                    med_ang_deg=("ang_deg_to_star", "median"),
                    max_ang_deg=("ang_deg_to_star", "max"),
                )
                per_country["flow"] = flow
                per_country["period"] = period_tag
                per_country["cluster"] = int(k)

                member_rows.extend(per_country.to_dict(orient="records"))

                star_lat, star_lon = unitvec_to_latlon_deg(star)

                star_rows.append({
                    "flow": flow,
                    "period": period_tag,
                    "cluster": int(k),
                    "year_min": int(y0),
                    "year_max": int(y1),
                    "n_countries": int(gk["country"].nunique()),
                    "n_obs": int(V.shape[0]),
                    "star_x": float(star[0]),
                    "star_y": float(star[1]),
                    "star_z": float(star[2]),
                    "star_lat": star_lat,
                    "star_lon": star_lon,
                    "mean_ang_deg_within": float(np.mean(ang_deg)),
                    "median_ang_deg_within": float(np.median(ang_deg)),
                    "p90_ang_deg_within": float(np.percentile(ang_deg, 90)),
                    "max_ang_deg_within": float(np.max(ang_deg)),
                })

            print(f"[OK] Stars computed for {flow} {period_tag} (clusters={len(clusters)})")

    stars = pd.DataFrame(star_rows).sort_values(["flow", "period", "cluster"])
    members = pd.DataFrame(member_rows).sort_values(
        ["flow", "period", "cluster", "mean_ang_deg"],
        ascending=[True, True, True, False]
    )

    stars.to_csv(OUT_STARS, index=False)
    members.to_csv(OUT_MEMBERS, index=False)

    print("\n✓ Wrote stars:")
    print(f"  {OUT_STARS}")
    print("✓ Wrote member diagnostics:")
    print(f"  {OUT_MEMBERS}")

if __name__ == "__main__":
    main()


[INFO] Using detected clustering SUFFIX = 20260129_020758
[OK] Stars computed for exports 1977_2000 (clusters=7)
[OK] Stars computed for exports 2001_2022 (clusters=5)
[OK] Stars computed for imports 1977_2000 (clusters=5)
[OK] Stars computed for imports 2001_2022 (clusters=3)

✓ Wrote stars:
  C:\Python\trade\geo\gravitationalcluster\gc_stars_20260129_020758.csv
✓ Wrote member diagnostics:
  C:\Python\trade\geo\gravitationalcluster\gc_star_members_20260129_020758.csv


**Geographic Stars**

In [35]:
# ============================================================
# Compute "Attractors / Stars" per (flow, period, cluster)
# READY TO RUN (Stars in RAW geography, Clusters from DEMEANED run)
#
# Inputs:
#   - gc_state_long_demeaned.csv  (must contain RAW x,y,z AND demeaned x_demean,y_demean,z_demean)
#   - gc_{flow}_{period}_{SUFFIX}_clusters.csv  (from your clustering run on demeaned vectors)
#
# Behavior:
#   - AUTO-DETECTS the most recent clustering SUFFIX in BASE_DIR
#   - Uses cluster labels from that run
#   - Computes stars using RAW x,y,z (for interpretability on globe)
#   - Also outputs per-country diagnostics: mean/median/max distance to RAW star
#
# Outputs:
#   - gc_stars_rawgeo_<SUFFIX>.csv
#   - gc_star_members_rawgeo_<SUFFIX>.csv
# ============================================================

import os
import re
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
STATE_FILE = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

FLOWS = ["exports", "imports"]
PERIODS = [
    ("1977_2000", 1977, 2000),
    ("2001_2022", 2001, 2022),
]

# ============================================================
# AUTO-DETECT MOST RECENT CLUSTERING SUFFIX
# ============================================================
def detect_latest_suffix(base_dir):
    """
    Finds the most recent gc_*_*_<SUFFIX>_clusters.csv file in base_dir
    and extracts <SUFFIX> of form YYYYMMDD_HHMMSS.
    """
    pat = re.compile(r"gc_(exports|imports)_(\d{4}_\d{4})_(\d{8}_\d{6})_clusters\.csv")
    suffixes = []
    for fn in os.listdir(base_dir):
        m = pat.match(fn)
        if m:
            suffixes.append(m.group(3))
    if not suffixes:
        raise FileNotFoundError(
            "No clustering files found in BASE_DIR. Expected files like:\n"
            "  gc_exports_1977_2000_YYYYMMDD_HHMMSS_clusters.csv"
        )
    return sorted(suffixes)[-1]

SUFFIX = detect_latest_suffix(BASE_DIR)
print(f"[INFO] Using detected clustering SUFFIX = {SUFFIX}")

OUT_STARS = os.path.join(BASE_DIR, f"gc_stars_rawgeo_{SUFFIX}.csv")
OUT_MEMBERS = os.path.join(BASE_DIR, f"gc_star_members_rawgeo_{SUFFIX}.csv")


# ============================================================
# GEOMETRY HELPERS
# ============================================================
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def unitvec_to_latlon_deg(v):
    """
    v: (3,) unit vector in R^3
    returns (lat_deg, lon_deg) with lon in (-180, 180]
    """
    x, y, z = float(v[0]), float(v[1]), float(v[2])
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    if lon <= -180:
        lon += 360
    elif lon > 180:
        lon -= 360
    return float(lat), float(lon)

def spherical_mean_unitvec(V):
    """
    V: (n,3) array of unit-ish vectors (already filtered finite)
    returns unit vector (3,)
    """
    s = np.sum(V, axis=0)
    n = np.linalg.norm(s)
    if not np.isfinite(n) or n == 0:
        return np.array([np.nan, np.nan, np.nan], dtype=float)
    return s / n

def angular_distance_deg(V, star):
    """
    V: (n,3) unit vectors; star: (3,) unit vector
    returns angular distances in degrees
    """
    dots = np.einsum("ij,j->i", V, star)
    return np.degrees(arccos_clip(dots))


# ============================================================
# IO HELPERS
# ============================================================
def clusters_path(flow, period_tag):
    return os.path.join(BASE_DIR, f"gc_{flow}_{period_tag}_{SUFFIX}_clusters.csv")

def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")


# ============================================================
# MAIN
# ============================================================
def main():
    assert_exists(STATE_FILE)

    state = pd.read_csv(STATE_FILE)
    state["year"] = state["year"].astype(int)

    # We REQUIRE raw x,y,z for raw-geo stars.
    # If your file only has x_demean,y_demean,z_demean, you must merge raw vectors in first.
    required = {"country", "flow", "year", "x", "y", "z"}
    missing = required - set(state.columns)
    if missing:
        raise ValueError(
            f"{STATE_FILE} is missing RAW columns: {sorted(missing)}.\n"
            "To compute raw-geo stars, the state file must contain x,y,z.\n"
            "If you only have x_demean/y_demean/z_demean, recompute or merge raw vectors."
        )

    state = state[["country", "flow", "year", "x", "y", "z"]].copy()

    star_rows = []
    member_rows = []

    for flow in FLOWS:
        sf = state[state["flow"] == flow].copy()
        if sf.empty:
            print(f"[WARN] No rows for flow={flow} in state file.")
            continue

        for period_tag, y0, y1 in PERIODS:
            cl_path = clusters_path(flow, period_tag)
            assert_exists(cl_path)
            cl = pd.read_csv(cl_path)

            reqc = {"country", "cluster"}
            missingc = reqc - set(cl.columns)
            if missingc:
                raise ValueError(f"{cl_path} missing columns: {sorted(missingc)}")

            # Window data (RAW vectors)
            dfp = sf[(sf["year"] >= y0) & (sf["year"] <= y1)].copy()

            # Attach clusters (clusters were computed on demeaned, but labels are by country)
            dfm = dfp.merge(cl[["country", "cluster"]], on="country", how="inner")
            if dfm.empty:
                print(f"[WARN] Empty merged data for {flow} {period_tag}.")
                continue

            # Keep finite raw vectors
            ok = np.isfinite(dfm[["x", "y", "z"]]).all(axis=1)
            dfm = dfm[ok].copy()
            if dfm.empty:
                print(f"[WARN] No finite RAW vectors for {flow} {period_tag}.")
                continue

            clusters = sorted(dfm["cluster"].unique())
            for k in clusters:
                gk = dfm[dfm["cluster"] == k].copy()
                if gk.empty:
                    continue

                V = gk[["x", "y", "z"]].to_numpy(dtype=float)

                # Normalize defensively (raw vectors should already be unit vectors)
                norms = np.linalg.norm(V, axis=1)
                keep = np.isfinite(norms) & (norms > 0)
                V = V[keep]
                if V.shape[0] == 0:
                    continue
                V = V / norms[keep][:, None]

                star = spherical_mean_unitvec(V)
                if not np.isfinite(star).all():
                    continue

                ang_deg = angular_distance_deg(V, star)

                # Per-country diagnostics (distance to RAW star)
                gk2 = gk.iloc[np.where(keep)[0]].copy()
                gk2["ang_deg_to_star_raw"] = ang_deg

                per_country = gk2.groupby("country", as_index=False).agg(
                    n_obs=("ang_deg_to_star_raw", "size"),
                    mean_ang_deg=("ang_deg_to_star_raw", "mean"),
                    med_ang_deg=("ang_deg_to_star_raw", "median"),
                    max_ang_deg=("ang_deg_to_star_raw", "max"),
                )
                per_country["flow"] = flow
                per_country["period"] = period_tag
                per_country["cluster"] = int(k)

                member_rows.extend(per_country.to_dict(orient="records"))

                star_lat, star_lon = unitvec_to_latlon_deg(star)

                star_rows.append({
                    "flow": flow,
                    "period": period_tag,
                    "cluster": int(k),
                    "year_min": int(y0),
                    "year_max": int(y1),
                    "n_countries": int(gk["country"].nunique()),
                    "n_obs": int(V.shape[0]),
                    "star_x": float(star[0]),
                    "star_y": float(star[1]),
                    "star_z": float(star[2]),
                    "star_lat": star_lat,
                    "star_lon": star_lon,
                    "mean_ang_deg_within": float(np.mean(ang_deg)),
                    "median_ang_deg_within": float(np.median(ang_deg)),
                    "p90_ang_deg_within": float(np.percentile(ang_deg, 90)),
                    "max_ang_deg_within": float(np.max(ang_deg)),
                })

            print(f"[OK] RAW-GEO Stars computed for {flow} {period_tag} (clusters={len(clusters)})")

    stars = pd.DataFrame(star_rows).sort_values(["flow", "period", "cluster"])
    members = pd.DataFrame(member_rows).sort_values(
        ["flow", "period", "cluster", "mean_ang_deg"],
        ascending=[True, True, True, False]
    )

    stars.to_csv(OUT_STARS, index=False)
    members.to_csv(OUT_MEMBERS, index=False)

    print("\n✓ Wrote RAW-GEO stars:")
    print(f"  {OUT_STARS}")
    print("✓ Wrote member diagnostics (RAW-GEO distances):")
    print(f"  {OUT_MEMBERS}")

if __name__ == "__main__":
    main()


[INFO] Using detected clustering SUFFIX = 20260129_020758
[OK] RAW-GEO Stars computed for exports 1977_2000 (clusters=7)
[OK] RAW-GEO Stars computed for exports 2001_2022 (clusters=5)
[OK] RAW-GEO Stars computed for imports 1977_2000 (clusters=5)
[OK] RAW-GEO Stars computed for imports 2001_2022 (clusters=3)

✓ Wrote RAW-GEO stars:
  C:\Python\trade\geo\gravitationalcluster\gc_stars_rawgeo_20260129_020758.csv
✓ Wrote member diagnostics (RAW-GEO distances):
  C:\Python\trade\geo\gravitationalcluster\gc_star_members_rawgeo_20260129_020758.csv


**Vizualization**

In [36]:
# ============================================================
# Publish-ready cluster maps (Mercator EPSG:3857), no Antarctica
#
# What this script does:
# - Loads Natural Earth countries shapefile (EPSG:4326) -> projects to EPSG:3857
# - Excludes Antarctica (ISO_A3 == "ATA")
# - Loads latest clustering run (auto-detect SUFFIX)
# - Loads stars (raw-geo) for that same SUFFIX
# - Produces 4 PNGs (exports/imports × 2 periods)
#
# Style goals (publish-ready):
# - Neutral basemap: very light gray fill + subtle borders
# - Clusters: qualitative, professional palettes (Okabe-Ito + Tableau-like extensions)
# - Stars: strong but not neon; consistent per flow (export red, import blue)
# - Labels: clean white halo box with subtle border; offset to reduce overlap
# - No legends by default (maps stay clean). Easy to add if needed.
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
SHAPEFILE = r"C:\Python\trade\geo\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"

FLOWS = ["exports", "imports"]
PERIODS = [
    ("1977_2000", 1977, 2000),
    ("2001_2022", 2001, 2022),
]

# Stars file produced earlier
STARS_PREFIX = "gc_stars_rawgeo_"  # expects gc_stars_rawgeo_<SUFFIX>.csv

# Output
OUT_DIR = BASE_DIR
DPI = 300

# Basemap styling (subtle, print-friendly)
BASEMAP_FACE = "#f2f2f2"
BASEMAP_EDGE = "#cfcfcf"
BASEMAP_EDGE_W = 0.35

# Cluster borders (very subtle)
CLUSTER_EDGE = "#ffffff"
CLUSTER_EDGE_W = 0.30

# Star styling (print-safe)
EXPORT_STAR_COLOR = "#b2182b"  # deep red
IMPORT_STAR_COLOR = "#2166ac"  # deep blue
STAR_SIZE = 55
STAR_EDGE = "white"
STAR_EDGE_W = 0.9

# Label styling
LABEL_FONT = 9
LABEL_BBOX = dict(boxstyle="round,pad=0.22", fc="white", ec="#444444", lw=0.4, alpha=0.92)

# Label offset in meters (Mercator): move slightly NE to avoid sitting on star
LABEL_DX = 140000
LABEL_DY = 90000

# ----------------------------
# Professional qualitative palettes
# ----------------------------
# Okabe-Ito (colorblind-safe) + a few Tableau-like additions to handle >8 clusters.
# This is a pragmatic "publishable" set: distinct, not neon, works on print.
OKABE_ITO = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # green
    "#D55E00",  # vermillion
    "#CC79A7",  # purple
    "#56B4E9",  # sky blue
    "#F0E442",  # yellow
    "#000000",  # black (use sparingly)
]
TABLEAU_EXT = [
    "#4E79A7",  # tableau blue
    "#F28E2B",  # tableau orange
    "#59A14F",  # tableau green
    "#E15759",  # tableau red
    "#B07AA1",  # tableau purple
    "#9C755F",  # tableau brown
    "#76B7B2",  # tableau teal
    "#EDC948",  # tableau yellow
    "#BAB0AC",  # tableau gray
]

# Exports palette and Imports palette: distinct “families” but both professional
EXPORT_PALETTE = (OKABE_ITO[:-1] + TABLEAU_EXT)  # avoid pure black for fills
IMPORT_PALETTE = (TABLEAU_EXT + OKABE_ITO[:-1])

# ----------------------------
# Helpers
# ----------------------------
def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

def detect_latest_suffix(base_dir):
    pat = re.compile(r"gc_(exports|imports)_(\d{4}_\d{4})_(\d{8}_\d{6})_clusters\.csv")
    suffixes = []
    for fn in os.listdir(base_dir):
        m = pat.match(fn)
        if m:
            suffixes.append(m.group(3))
    if not suffixes:
        raise FileNotFoundError(
            "No clustering files found. Expected files like:\n"
            "  gc_exports_1977_2000_YYYYMMDD_HHMMSS_clusters.csv"
        )
    return sorted(suffixes)[-1]

def clusters_path(flow, period_tag, suffix):
    return os.path.join(BASE_DIR, f"gc_{flow}_{period_tag}_{suffix}_clusters.csv")

def stars_path(suffix):
    return os.path.join(BASE_DIR, f"{STARS_PREFIX}{suffix}.csv")

def make_cmap_for_clusters(flow, n):
    palette = EXPORT_PALETTE if flow == "exports" else IMPORT_PALETTE
    if n <= len(palette):
        cols = palette[:n]
    else:
        # If you somehow have > ~16 clusters, cycle (still distinct-ish but not perfect).
        cols = [palette[i % len(palette)] for i in range(n)]
    return ListedColormap(cols)

def plot_one(world_3857, clusters_df, stars_df, flow, period_tag, suffix):
    # Prepare base layer
    gdf = world_3857.copy()
    gdf = gdf[gdf["ISO_A3"].notna() & (gdf["ISO_A3"] != "-99")].copy()
    gdf = gdf[gdf["ISO_A3"] != "ATA"].copy()

    # Merge clusters by ISO3
    cl = clusters_df.copy()
    cl["country"] = cl["country"].astype(str).str.upper()
    gdf = gdf.merge(cl[["country", "cluster"]], left_on="ISO_A3", right_on="country", how="left")

    # Present clusters and mapping to indices
    present = np.sort(gdf["cluster"].dropna().astype(int).unique())
    k = len(present)
    idx_map = {c: i for i, c in enumerate(present)}
    gdf["cidx"] = gdf["cluster"].map(lambda x: idx_map.get(int(x)) if pd.notna(x) else np.nan)

    cmap = make_cmap_for_clusters(flow, max(k, 1))
    star_color = EXPORT_STAR_COLOR if flow == "exports" else IMPORT_STAR_COLOR

    # Create figure (clean)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_axis_off()

    # Basemap
    gdf.plot(
        ax=ax,
        color=BASEMAP_FACE,
        edgecolor=BASEMAP_EDGE,
        linewidth=BASEMAP_EDGE_W,
        zorder=1
    )

    # Cluster fill
    clustered = gdf[gdf["cidx"].notna()].copy()
    if not clustered.empty:
        clustered.plot(
            ax=ax,
            column="cidx",
            cmap=cmap,
            edgecolor=CLUSTER_EDGE,
            linewidth=CLUSTER_EDGE_W,
            zorder=2
        )

    # Stars (raw-geo lat/lon -> EPSG:3857)
    st = stars_df[(stars_df["flow"] == flow) & (stars_df["period"] == period_tag)].copy()
    if not st.empty:
        st_g = gpd.GeoDataFrame(
            st,
            geometry=gpd.points_from_xy(st["star_lon"], st["star_lat"]),
            crs="EPSG:4326"
        ).to_crs("EPSG:3857")

        st_g.plot(
            ax=ax,
            color=star_color,
            markersize=STAR_SIZE,
            edgecolor=STAR_EDGE,
            linewidth=STAR_EDGE_W,
            zorder=5
        )

        # Labels: n_countries, slightly offset for readability
        for r in st_g.itertuples():
            x = r.geometry.x + LABEL_DX
            y = r.geometry.y + LABEL_DY
            ax.text(
                x, y,
                str(int(r.n_countries)) if pd.notna(r.n_countries) else "",
                fontsize=LABEL_FONT,
                ha="center",
                va="center",
                bbox=LABEL_BBOX,
                zorder=6
            )

    # Title: restrained, publication style
    ax.set_title(f"{flow.capitalize()} clusters, {period_tag}", fontsize=14)

    out = os.path.join(OUT_DIR, f"gc_map_{flow}_{period_tag}_{suffix}_PUB.png")
    plt.tight_layout()
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved: {out}")

# ----------------------------
# MAIN
# ----------------------------
def main():
    assert_exists(SHAPEFILE)

    suffix = detect_latest_suffix(BASE_DIR)
    print(f"[INFO] Using clustering suffix: {suffix}")

    spath = stars_path(suffix)
    assert_exists(spath)

    stars = pd.read_csv(spath)
    req_st = {"flow", "period", "cluster", "star_lat", "star_lon", "n_countries"}
    miss = req_st - set(stars.columns)
    if miss:
        raise ValueError(f"{spath} missing columns: {sorted(miss)}")

    world = gpd.read_file(SHAPEFILE)
    if world.crs is None:
        world = world.set_crs("EPSG:4326")
    else:
        world = world.to_crs("EPSG:4326")
    world_3857 = world.to_crs("EPSG:3857")

    for flow in FLOWS:
        for period_tag, _, _ in PERIODS:
            cpath = clusters_path(flow, period_tag, suffix)
            assert_exists(cpath)
            clusters = pd.read_csv(cpath)
            if not {"country", "cluster"}.issubset(clusters.columns):
                raise ValueError(f"{cpath} must contain at least ['country','cluster']")


**Huricaine Baricenter**

In [37]:
# ============================================================
# Mercator maps of country barycenters ("stars") for ALL countries
# - One map for exports, one map for imports
# - No labels
# - Basemap: Natural Earth countries, gray, no Antarctica
# - Points: spherical barycenter per country (across ALL years in the file)
#
# Inputs:
#   1) STATE_FILE: gc_state_long_demeaned.csv (must contain raw x,y,z OR demeaned x_demean,y_demean,z_demean)
#   2) SHAPEFILE:  Natural Earth admin_0 countries shapefile (EPSG:4326)
#
# Output:
#   - gc_country_barycenters_exports.png
#   - gc_country_barycenters_imports.png
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
STATE_FILE = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")

SHAPEFILE = r"C:\Python\trade\geo\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"

FLOWS = ["exports", "imports"]

# Use RAW for interpretable geography. If you only have demeaned, set True.
USE_DEMEANED = False  # <-- set True ONLY if raw x,y,z are not available

# Basemap styling
BASEMAP_FACE = "#f2f2f2"
BASEMAP_EDGE = "#cfcfcf"
BASEMAP_EDGE_W = 0.35

# Point styling (publish-friendly, no labels)
EXPORT_POINT_COLOR = "#b2182b"  # deep red
IMPORT_POINT_COLOR = "#2166ac"  # deep blue
POINT_SIZE = 10
POINT_ALPHA = 0.85
POINT_EDGE = "white"
POINT_EDGE_W = 0.25

DPI = 300

# ----------------------------
# Geometry helpers
# ----------------------------
def arccos_clip(x):
    return np.arccos(np.clip(x, -1.0, 1.0))

def spherical_mean_unitvec(V):
    s = np.sum(V, axis=0)
    n = np.linalg.norm(s)
    if not np.isfinite(n) or n == 0:
        return np.array([np.nan, np.nan, np.nan], dtype=float)
    return s / n

def unitvec_to_latlon_deg(v):
    x, y, z = float(v[0]), float(v[1]), float(v[2])
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    if lon <= -180:
        lon += 360
    elif lon > 180:
        lon -= 360
    return float(lat), float(lon)

def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

# ----------------------------
# Compute barycenters per country
# ----------------------------
def compute_country_barycenters(state_df, flow):
    df = state_df[state_df["flow"] == flow].copy()
    if df.empty:
        return pd.DataFrame(columns=["country", "lat", "lon"])

    # Drop non-finite vectors
    ok = np.isfinite(df[["x", "y", "z"]]).all(axis=1)
    df = df[ok].copy()
    if df.empty:
        return pd.DataFrame(columns=["country", "lat", "lon"])

    out = []
    for c, gc in df.groupby("country", sort=True):
        V = gc[["x", "y", "z"]].to_numpy(dtype=float)

        # normalize rows defensively
        norms = np.linalg.norm(V, axis=1)
        keep = np.isfinite(norms) & (norms > 0)
        V = V[keep]
        if V.shape[0] == 0:
            continue
        V = V / norms[keep][:, None]

        star = spherical_mean_unitvec(V)
        if not np.isfinite(star).all():
            continue

        lat, lon = unitvec_to_latlon_deg(star)
        out.append({"country": c, "lat": lat, "lon": lon})

    return pd.DataFrame(out)

# ----------------------------
# Plot map
# ----------------------------
def plot_barycenters(world_3857, pts_ll, flow):
    pts = gpd.GeoDataFrame(
        pts_ll,
        geometry=gpd.points_from_xy(pts_ll["lon"], pts_ll["lat"]),
        crs="EPSG:4326",
    ).to_crs("EPSG:3857")

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_axis_off()

    # Basemap (no Antarctica)
    world_3857.plot(
        ax=ax,
        color=BASEMAP_FACE,
        edgecolor=BASEMAP_EDGE,
        linewidth=BASEMAP_EDGE_W,
        zorder=1
    )

    color = EXPORT_POINT_COLOR if flow == "exports" else IMPORT_POINT_COLOR

    pts.plot(
        ax=ax,
        color=color,
        markersize=POINT_SIZE,
        alpha=POINT_ALPHA,
        edgecolor=POINT_EDGE,
        linewidth=POINT_EDGE_W,
        zorder=5
    )

    ax.set_title(f"{flow.capitalize()} country barycenters (Mercator)", fontsize=14)

    out = os.path.join(BASE_DIR, f"gc_country_barycenters_{flow}.png")
    plt.tight_layout()
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"✓ Saved: {out}")

# ----------------------------
# MAIN
# ----------------------------
def main():
    assert_exists(STATE_FILE)
    assert_exists(SHAPEFILE)

    state = pd.read_csv(STATE_FILE)
    state["year"] = state["year"].astype(int)

    if USE_DEMEANED:
        needed = {"country", "flow", "year", "x_demean", "y_demean", "z_demean"}
        miss = needed - set(state.columns)
        if miss:
            raise ValueError(f"{STATE_FILE} missing columns: {sorted(miss)}")
        state = state.rename(columns={"x_demean": "x", "y_demean": "y", "z_demean": "z"})
    else:
        needed = {"country", "flow", "year", "x", "y", "z"}
        miss = needed - set(state.columns)
        if miss:
            raise ValueError(
                f"{STATE_FILE} missing RAW columns: {sorted(miss)}.\n"
                "Set USE_DEMEANED=True only if you intentionally want demeaned barycenters."
            )

    state = state[["country", "flow", "year", "x", "y", "z"]].copy()

    world = gpd.read_file(SHAPEFILE)
    if world.crs is None:
        world = world.set_crs("EPSG:4326")
    else:
        world = world.to_crs("EPSG:4326")

    # Remove Antarctica and junk codes, then project to Mercator
    world = world[world["ISO_A3"].notna() & (world["ISO_A3"] != "-99")].copy()
    world = world[world["ISO_A3"] != "ATA"].copy()
    world_3857 = world.to_crs("EPSG:3857")

    for flow in FLOWS:
        pts = compute_country_barycenters(state, flow)
        if pts.empty:
            print(f"[WARN] No barycenters computed for flow={flow}.")
            continue
        plot_barycenters(world_3857, pts, flow)

    print("\nDone.")

if __name__ == "__main__":
    main()


✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_country_barycenters_exports.png
✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_country_barycenters_imports.png

Done.


In [2]:
# ============================================================
# Country barycenter trajectories (year-to-year great-circle paths)
# - Two maps: exports and imports
# - Mercator EPSG:3857, no Antarctica
# - Trajectories are drawn as great-circle interpolations between yearly barycenters
#
# Inputs:
#   - gc_state_long_demeaned.csv  (must contain raw x,y,z or demeaned)
#   - Natural Earth shapefile (EPSG:4326)
#
# Outputs:
#   - gc_trajectories_exports.png
#   - gc_trajectories_imports.png
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

# ----------------------------
# CONFIG
# ----------------------------
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
STATE_FILE = os.path.join(BASE_DIR, "gc_state_long_demeaned.csv")
SHAPEFILE = r"C:\Python\trade\geo\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"

FLOWS = ["exports", "imports"]

# Use raw geography vectors (recommended). Set True only if raw x,y,z missing.
USE_DEMEANED = False

# Years to plot (inclusive). Use None to use full available range.
YEAR_MIN = 1980
YEAR_MAX = 2022

# Trajectory detail: points per segment (year->year). 20 is smooth enough.
PTS_PER_SEGMENT = 20

# Plot control to avoid spaghetti:
# - "all": draw all countries
# - "topN": draw only top N countries by number of years present
# - "subset": draw only ISO3 in SELECT_COUNTRIES
MODE = "all"
TOP_N = 60
SELECT_COUNTRIES = ["USA", "CHN", "DEU", "JPN", "MEX", "BRA", "IND"]

# Basemap styling
BASEMAP_FACE = "#f2f2f2"
BASEMAP_EDGE = "#cfcfcf"
BASEMAP_EDGE_W = 0.35

# Line styling (publish-friendly)
EXPORT_LINE = "#b2182b"  # deep red
IMPORT_LINE = "#2166ac"  # deep blue
LINE_W = 0.55
LINE_ALPHA = 0.20

# Start/end markers (optional; keep subtle)
DRAW_ENDPOINTS = True
START_MARKER = "o"
END_MARKER = "s"
MARKER_SIZE = 8
MARKER_ALPHA = 0.35
MARKER_EDGE = "white"
MARKER_EDGE_W = 0.25

DPI = 300

# ----------------------------
# Helpers (spherical interpolation)
# ----------------------------
def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

def _normalize_rows(V):
    n = np.linalg.norm(V, axis=1)
    keep = np.isfinite(n) & (n > 0)
    V2 = V[keep] / n[keep][:, None]
    return V2, keep

def slerp(u, v, t):
    """
    Spherical linear interpolation between unit vectors u and v.
    u, v: (3,) unit vectors
    t: float in [0,1]
    returns: (3,) unit vector
    """
    dot = float(np.clip(np.dot(u, v), -1.0, 1.0))
    omega = np.arccos(dot)
    if omega < 1e-12:
        return u.copy()
    so = np.sin(omega)
    return (np.sin((1 - t) * omega) / so) * u + (np.sin(t * omega) / so) * v

def unitvec_to_lonlat_deg(p):
    x, y, z = float(p[0]), float(p[1]), float(p[2])
    lat = np.degrees(np.arcsin(np.clip(z, -1.0, 1.0)))
    lon = np.degrees(np.arctan2(y, x))
    # normalize lon to (-180, 180]
    if lon <= -180:
        lon += 360
    elif lon > 180:
        lon -= 360
    return lon, lat

def build_country_lines(df_country, pts_per_segment=PTS_PER_SEGMENT):
    """
    df_country: rows for a single country+flow with columns year,x,y,z (unit-ish)
    returns: list of shapely LineString in lon/lat
    """
    dfc = df_country.sort_values("year")
    V = dfc[["x","y","z"]].to_numpy(float)
    Vn, keep = _normalize_rows(V)
    if Vn.shape[0] < 2:
        return [], None, None

    years = dfc["year"].to_numpy(int)[keep]
    # Build segments only where consecutive years exist (t -> t+1)
    # If you want to connect across gaps, change this logic.
    lines = []
    start_ll = None
    end_ll = None

    for i in range(len(years) - 1):
        y0, y1 = years[i], years[i+1]
        if y1 != y0 + 1:
            continue  # skip gaps; avoids long jumps across missing years
        u = Vn[i]
        v = Vn[i+1]
        pts = []
        for k in range(pts_per_segment + 1):
            t = k / pts_per_segment
            p = slerp(u, v, t)
            lon, lat = unitvec_to_lonlat_deg(p)
            pts.append((lon, lat))
        if not start_ll:
            start_ll = pts[0]
        end_ll = pts[-1]
        lines.append(LineString(pts))

    return lines, start_ll, end_ll

# ----------------------------
# MAIN
# ----------------------------
def main():
    assert_exists(STATE_FILE)
    assert_exists(SHAPEFILE)

    state = pd.read_csv(STATE_FILE)
    state["year"] = state["year"].astype(int)

    if USE_DEMEANED:
        needed = {"country","flow","year","x_demean","y_demean","z_demean"}
        miss = needed - set(state.columns)
        if miss:
            raise ValueError(f"{STATE_FILE} missing columns: {sorted(miss)}")
        state = state.rename(columns={"x_demean":"x","y_demean":"y","z_demean":"z"})
    else:
        needed = {"country","flow","year","x","y","z"}
        miss = needed - set(state.columns)
        if miss:
            raise ValueError(
                f"{STATE_FILE} missing RAW columns: {sorted(miss)}.\n"
                "Set USE_DEMEANED=True only if you intentionally want demeaned trajectories."
            )

    state = state[["country","flow","year","x","y","z"]].copy()
    state["country"] = state["country"].astype(str).str.upper()

    # Filter years
    if YEAR_MIN is not None:
        state = state[state["year"] >= YEAR_MIN].copy()
    if YEAR_MAX is not None:
        state = state[state["year"] <= YEAR_MAX].copy()

    # Basemap -> Mercator
    world = gpd.read_file(SHAPEFILE)
    if world.crs is None:
        world = world.set_crs("EPSG:4326")
    else:
        world = world.to_crs("EPSG:4326")
    world = world[world["ISO_A3"].notna() & (world["ISO_A3"] != "-99")].copy()
    world = world[world["ISO_A3"] != "ATA"].copy()
    world_3857 = world.to_crs("EPSG:3857")

    for flow in FLOWS:
        df = state[state["flow"] == flow].copy()
        if df.empty:
            print(f"[WARN] No data for flow={flow}")
            continue

        # choose which countries to draw
        if MODE == "subset":
            use_countries = sorted(set([c.upper() for c in SELECT_COUNTRIES]))
        else:
            counts = df.groupby("country")["year"].nunique().sort_values(ascending=False)
            if MODE == "topN":
                use_countries = counts.head(TOP_N).index.tolist()
            else:
                use_countries = counts.index.tolist()

        line_geoms = []
        start_pts = []
        end_pts = []

        for c in use_countries:
            dfc = df[df["country"] == c]
            if dfc.shape[0] < 2:
                continue
            lines, start_ll, end_ll = build_country_lines(dfc, pts_per_segment=PTS_PER_SEGMENT)
            if not lines:
                continue
            line_geoms.extend(lines)
            if DRAW_ENDPOINTS and start_ll and end_ll:
                start_pts.append(start_ll)
                end_pts.append(end_ll)

        if not line_geoms:
            print(f"[WARN] No trajectories drawn for flow={flow} (check YEAR range / missing data).")
            continue

        lines_gdf = gpd.GeoDataFrame(geometry=line_geoms, crs="EPSG:4326").to_crs("EPSG:3857")

        # Endpoints (optional)
        if DRAW_ENDPOINTS:
            start_gdf = gpd.GeoDataFrame(
                geometry=gpd.points_from_xy([p[0] for p in start_pts], [p[1] for p in start_pts]),
                crs="EPSG:4326"
            ).to_crs("EPSG:3857") if start_pts else None

            end_gdf = gpd.GeoDataFrame(
                geometry=gpd.points_from_xy([p[0] for p in end_pts], [p[1] for p in end_pts]),
                crs="EPSG:4326"
            ).to_crs("EPSG:3857") if end_pts else None
        else:
            start_gdf = end_gdf = None

        # Plot
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.set_axis_off()

        world_3857.plot(ax=ax, color=BASEMAP_FACE, edgecolor=BASEMAP_EDGE,
                        linewidth=BASEMAP_EDGE_W, zorder=1)

        line_color = EXPORT_LINE if flow == "exports" else IMPORT_LINE
        lines_gdf.plot(ax=ax, color=line_color, linewidth=LINE_W, alpha=LINE_ALPHA, zorder=3)

        if DRAW_ENDPOINTS and start_gdf is not None and end_gdf is not None:
            start_gdf.plot(ax=ax, color=line_color, markersize=MARKER_SIZE, alpha=MARKER_ALPHA,
                           edgecolor=MARKER_EDGE, linewidth=MARKER_EDGE_W, zorder=4, marker=START_MARKER)
            end_gdf.plot(ax=ax, color=line_color, markersize=MARKER_SIZE, alpha=MARKER_ALPHA,
                         edgecolor=MARKER_EDGE, linewidth=MARKER_EDGE_W, zorder=4, marker=END_MARKER)

        ax.set_title(
            f"{flow.capitalize()} barycenter trajectories ({YEAR_MIN}–{YEAR_MAX}) — {MODE}",
            fontsize=14
        )

        out = os.path.join(BASE_DIR, f"gc_trajectories_{flow}.png")
        plt.tight_layout()
        plt.savefig(out, dpi=DPI, bbox_inches="tight")
        plt.close(fig)

        print(f"✓ Saved: {out}  (countries drawn: {len(use_countries)}, segments: {len(line_geoms)})")

    print("\nDone.")

if __name__ == "__main__":
    main()


✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_trajectories_exports.png  (countries drawn: 249, segments: 9196)
✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_trajectories_imports.png  (countries drawn: 249, segments: 9190)

Done.


In [4]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from matplotlib.lines import Line2D

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Python\trade\geo\gravitationalcluster"
SHAPEFILE = r"C:\Python\trade\geo\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"

# Your per-country files:
#   barycenter_ABW_1977_2022.csv, barycenter_USA_1977_2022.csv, ...
BARY_GLOB = os.path.join(BASE_DIR, "barycenter_*_1977_2022*.csv")

YEAR_MIN = 1977
YEAR_MAX = 2022

# East/West classification threshold per year (degrees longitude)
DLON_THRESH_DEG = 2.0

# Optional densification of each yearly segment in lon/lat (0 = none)
PTS_PER_SEGMENT = 0

# Basemap
BASEMAP_FACE = "#f2f2f2"
BASEMAP_EDGE = "#cfcfcf"
BASEMAP_EDGE_W = 0.35

# Trajectories
LINE_W = 0.55
ALPHA = 0.22
COLOR_EAST = "#b2182b"   # red
COLOR_WEST = "#2166ac"   # blue
COLOR_UNCL = "#7f7f7f"   # gray

DPI = 300

# Projection
PROJ_PRIMARY = "ESRI:54030"   # Robinson
PROJ_FALLBACK = "ESRI:54009"  # Mollweide

# Output
OUT_EXPORTS = os.path.join(BASE_DIR, "gc_trajectories_exports_publish.png")
OUT_IMPORTS = os.path.join(BASE_DIR, "gc_trajectories_imports_publish.png")


# ============================================================
# HELPERS
# ============================================================
def assert_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

def pick_projection(world_gdf):
    try:
        world_gdf.to_crs(PROJ_PRIMARY)
        return PROJ_PRIMARY
    except Exception:
        return PROJ_FALLBACK

def unwrap_lon(lon_deg_array):
    lon_rad = np.deg2rad(lon_deg_array.astype(float))
    lon_unw = np.unwrap(lon_rad)
    return np.rad2deg(lon_unw)

def densify_segment(lon0, lat0, lon1, lat1, n=0):
    if not n or n <= 0:
        return [(lon0, lat0), (lon1, lat1)]
    xs = np.linspace(lon0, lon1, n + 2)
    ys = np.linspace(lat0, lat1, n + 2)
    return list(zip(xs, ys))

def read_country_barycenter(path):
    """
    Reads your schema:
      country, lat_exports, lon_exports, exports_weight,
              lat_imports, lon_imports, imports_weight, year
    Returns a dataframe with year and the relevant lon/lat columns for both flows.
    """
    df = pd.read_csv(path)

    required = {
        "country", "year",
        "lat_exports", "lon_exports",
        "lat_imports", "lon_imports",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{os.path.basename(path)} missing columns: {sorted(missing)}")

    df = df.copy()
    df["year"] = df["year"].astype(int)

    for c in ["lat_exports","lon_exports","lat_imports","lon_imports"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)].copy()
    return df

def build_segments_for_flow(df, flow):
    """
    flow in {"exports","imports"}
    Returns 3 lists of LineString in lon/lat: east, west, unclear
    """
    if flow == "exports":
        latcol, loncol = "lat_exports", "lon_exports"
    else:
        latcol, loncol = "lat_imports", "lon_imports"

    d = df[["year", loncol, latcol]].rename(columns={loncol: "lon", latcol: "lat"}).copy()
    d = d[np.isfinite(d["lon"]) & np.isfinite(d["lat"])].sort_values("year")

    if d.shape[0] < 2:
        return [], [], []

    years = d["year"].to_numpy(int)
    lons = d["lon"].to_numpy(float)
    lats = d["lat"].to_numpy(float)
    lons_unw = unwrap_lon(lons)

    east, west, uncl = [], [], []

    for i in range(len(years) - 1):
        y0, y1 = years[i], years[i + 1]
        if y1 != y0 + 1:
            continue  # skip gaps

        dlon = lons_unw[i + 1] - lons_unw[i]

        pts = densify_segment(lons[i], lats[i], lons[i + 1], lats[i + 1], n=PTS_PER_SEGMENT)
        seg = LineString(pts)

        if dlon > DLON_THRESH_DEG:
            east.append(seg)
        elif dlon < -DLON_THRESH_DEG:
            west.append(seg)
        else:
            uncl.append(seg)

    return east, west, uncl

def plot_flow(world_4326, flow, out_path):
    files = sorted(glob.glob(BARY_GLOB))
    print(f"[INFO] {flow}: found {len(files)} barycenter files matching {os.path.basename(BARY_GLOB)}")
    if not files:
        raise FileNotFoundError(f"No barycenter files found at: {BARY_GLOB}")

    east_lines, west_lines, uncl_lines = [], [], []

    for fp in files:
        # skip Antarctica barycenter if present
        if os.path.basename(fp).upper().startswith("BARYCENTER_ATA_"):
            continue

        try:
            df = read_country_barycenter(fp)
        except Exception as e:
            print(f"[WARN] Skipping {os.path.basename(fp)}: {e}")
            continue

        e, w, u = build_segments_for_flow(df, flow)
        east_lines.extend(e)
        west_lines.extend(w)
        uncl_lines.extend(u)

    if not (east_lines or west_lines or uncl_lines):
        raise ValueError(f"No valid segments constructed for {flow} in {YEAR_MIN}-{YEAR_MAX}.")

    proj = pick_projection(world_4326)

    world = world_4326.copy()
    world = world[world["ISO_A3"].notna() & (world["ISO_A3"] != "-99")].copy()
    world = world[world["ISO_A3"] != "ATA"].copy()
    world_p = world.to_crs(proj)

    def to_proj(lines):
        if not lines:
            return None
        g = gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326")
        return g.to_crs(proj)

    east_g = to_proj(east_lines)
    west_g = to_proj(west_lines)
    uncl_g = to_proj(uncl_lines)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_axis_off()

    world_p.plot(ax=ax, color=BASEMAP_FACE, edgecolor=BASEMAP_EDGE, linewidth=BASEMAP_EDGE_W, zorder=1)

    # draw unclear first, then directional
    if uncl_g is not None:
        uncl_g.plot(ax=ax, color=COLOR_UNCL, linewidth=LINE_W, alpha=ALPHA, zorder=2)
    if west_g is not None:
        west_g.plot(ax=ax, color=COLOR_WEST, linewidth=LINE_W, alpha=ALPHA, zorder=3)
    if east_g is not None:
        east_g.plot(ax=ax, color=COLOR_EAST, linewidth=LINE_W, alpha=ALPHA, zorder=4)

    # Legend (no title on map)
    legend_title = "Trajectory of Export Barycenters" if flow == "exports" else "Trajectory of Import Barycenters"
    handles = [
        Line2D([0], [0], color=COLOR_EAST, lw=2, label="Eastbound"),
        Line2D([0], [0], color=COLOR_WEST, lw=2, label="Westbound"),
        Line2D([0], [0], color=COLOR_UNCL, lw=2, label="Unclear"),
    ]
    ax.legend(
        handles=handles,
        title=legend_title,
        loc="lower left",
        frameon=True,
        framealpha=0.95,
        edgecolor="#cccccc",
        fontsize=9,
        title_fontsize=10
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print(f"✓ Saved: {out_path}")
    print(f"  segments: east={len(east_lines)}, west={len(west_lines)}, unclear={len(uncl_lines)}")
    print(f"  projection: {proj}")


# ============================================================
# RUN
# ============================================================
def main():
    assert_exists(SHAPEFILE)
    world = gpd.read_file(SHAPEFILE)
    if world.crs is None:
        world = world.set_crs("EPSG:4326")
    else:
        world = world.to_crs("EPSG:4326")

    plot_flow(world, "exports", OUT_EXPORTS)
    plot_flow(world, "imports", OUT_IMPORTS)

if __name__ == "__main__":
    main()


[INFO] exports: found 250 barycenter files matching barycenter_*_1977_2022*.csv
✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_trajectories_exports_publish.png
  segments: east=3396, west=3094, unclear=3263
  projection: ESRI:54030
[INFO] imports: found 250 barycenter files matching barycenter_*_1977_2022*.csv
✓ Saved: C:\Python\trade\geo\gravitationalcluster\gc_trajectories_imports_publish.png
  segments: east=3199, west=2727, unclear=3820
  projection: ESRI:54030
